# GreenLedger — Campus Resource Optimisation
### Innovators Conclave 2026 · IC26-05-01 · Sustainability & Climate Action

> *"Build a platform that combines periodic readings and action logs, highlights unusual changes and compares resource use against a baseline **without overstating environmental benefits**."*

Most campus dashboards show charts and claim savings from simple before/after comparisons. Those claims are often wrong, because the term ended, the weather changed, or the whole campus dipped. **GreenLedger counts a saving only when it survives a counterfactual test, and then only at its 95% lower bound.**

| Problem-statement requirement | Where it lives in GreenLedger |
|---|---|
| Combine electricity, water and waste | **Overview**: one dashboard, three resources, five buildings, weekly scorecard with significance colouring |
| Periodic readings + action logs | **Action Register**: log interventions (what, where, when, cost, claimed %); linked to verification |
| Highlight unusual changes | **Alerts**: spike and drift tests (electricity), **minimum-night-flow** leak test (water), daily volume (waste), each with a plain-English explanation and suggested check; staff confirm, dismiss or convert to action |
| Compare against a baseline **without overstating** | **Verify Savings**: comparison-group counterfactual + **placebo-in-time** 95% intervals; VERIFIED only if the whole interval is above zero, plus safeguards; non-routine (alert) days excluded |
| Exportable report | **Report**: printable HTML (→ PDF), Markdown, CSV ledger, JSON |
| Labelled sample dataset | **Evidence**: real meter data with injected anomalies and interventions with known truth, downloadable as a ZIP |

**Data.** Electricity and water are **real hourly meter readings** from the Building Data Genome 2 project (5 college-classroom buildings at one site, 2016–2017), with real weather. Dry-waste data are **synthetic** (BDG2 has none). Interventions and anomalies are **injected with known truth**, so every claim the platform makes can be scored.

**How to run on Kaggle:** attach the *Building Data Genome Project 2* dataset, turn **Internet ON** (for `pip` and the public demo link), then *Run All*. Startup takes about 5–8 minutes on CPU, because every action is verified and the verifier is validated before the app opens.

In [ ]:
!pip -q install lightgbm plotly gradio scipy

## 1 · Setup and configuration
Campus constants (emission factor, assumed tariff) live here. Edit them for a real deployment.

In [ ]:
# ============================================================
# 1 · SETUP & CONFIG
# ============================================================
import os, io, json, glob, time, math, sqlite3, random, zipfile, warnings, traceback
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import lightgbm as lgb
import plotly.graph_objects as go
from scipy import stats

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------- Paths (auto-discovers the BDG2 files on Kaggle or locally) ----------
SEARCH_ROOTS = [os.environ.get("GL_DATA_DIR", ""), "/kaggle/input", "./data", "."]
OUT_DIR = os.environ.get("GL_OUT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else "./greenledger_out")
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, "greenledger.db")


def find_file(name):
    """Find a BDG2 file anywhere under the search roots (Kaggle mounts vary)."""
    for root in SEARCH_ROOTS:
        if not root or not os.path.isdir(root):
            continue
        direct = os.path.join(root, name)
        if os.path.isfile(direct):
            return direct
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if hits:
            return sorted(hits, key=len)[0]
    return None


# ---------- Campus constants (edit these for a real deployment) ----------
EMISSION_FACTOR = 0.710        # kgCO2/kWh — CEA CO2 Baseline Database v21.0, FY2024-25 weighted avg (provisional)
EF_SOURCE = "CEA CO₂ Baseline Database v21.0 (FY2024-25, provisional) · 0.710 kgCO₂/kWh"
TARIFF_INR_PER_KWH = 9.0       # ASSUMED commercial tariff — replace with your campus tariff
N_BUILDINGS = 5

# ---------- Analysis windows ----------
TRAIN_END = pd.Timestamp("2017-06-30 23:00")    # anomaly models learn "normal" up to here
EVAL_START = pd.Timestamp("2017-07-01")          # monitoring / demo period starts here
MV_TRAIN_DAYS = 365                              # counterfactual baseline window (days before action)
MV_POST_DAYS_DEFAULT = 42                        # measurement window after action (6 weeks)
MV_PLACEBO_STEP = 5                              # days between placebo (fake) action dates
SEED = 42
np.random.seed(SEED); random.seed(SEED)

RESOURCES = {
    "electricity": {"label": "Electricity", "unit": "kWh", "daily_unit": "kWh", "color": "#10B981"},
    "water":       {"label": "Water",       "unit": "L",   "daily_unit": "L",   "color": "#3B82F6"},
    "waste":       {"label": "Dry waste (landfill)", "unit": "kg", "daily_unit": "kg", "color": "#F59E0B"},
}

LGB_HOURLY = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=20,
                  subsample=0.9, subsample_freq=1, colsample_bytree=0.9, verbosity=-1,
                  random_state=SEED, n_jobs=-1)
LGB_DAILY = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=7,
                 subsample=0.9, subsample_freq=1, colsample_bytree=0.9, verbosity=-1,
                 random_state=SEED, n_jobs=1)

print("Output dir:", OUT_DIR)

## 2 · Real meter data
Auto-selects the best-instrumented campus in Building Data Genome 2: education buildings with near-complete electricity **and** water meters.

In [ ]:
# ============================================================
# 2 · REAL METER DATA (BDG2) — auto-select the best-instrumented campus
# ============================================================
P_META, P_ELEC = find_file("metadata.csv"), find_file("electricity_cleaned.csv")
P_WATER, P_WX = find_file("water_cleaned.csv"), find_file("weather.csv")
assert P_META and P_ELEC and P_WATER, "BDG2 files not found — attach the Building Data Genome 2 dataset."
print("metadata:", P_META, "\nelectricity:", P_ELEC, "\nwater:", P_WATER, "\nweather:", P_WX)

meta = pd.read_csv(P_META)
elec_cols = pd.read_csv(P_ELEC, nrows=0).columns.tolist()
water_cols = pd.read_csv(P_WATER, nrows=0).columns.tolist()
edu = meta[(meta.primaryspaceusage == "Education")
           & meta.building_id.isin(elec_cols) & meta.building_id.isin(water_cols)]
cand = edu.building_id.tolist()

_e = pd.read_csv(P_ELEC, usecols=["timestamp"] + cand, parse_dates=["timestamp"], index_col="timestamp")
_w = pd.read_csv(P_WATER, usecols=["timestamp"] + cand, parse_dates=["timestamp"], index_col="timestamp")
score = pd.Series({b: min(_e[b].notna().mean(), (_w[b] > 0).mean()) for b in cand})
edu = edu.assign(score=edu.building_id.map(score))
site_rank = (edu.sort_values("score", ascending=False).groupby("site_id")
             .apply(lambda g: g.score.head(N_BUILDINGS).mean() if len(g) >= N_BUILDINGS else 0))
SITE = site_rank.idxmax()
picked = edu[edu.site_id == SITE].sort_values("score", ascending=False).head(N_BUILDINGS)
BDG_IDS = picked.building_id.tolist()
BLDS = [f"Block {chr(65 + i)}" for i in range(len(BDG_IDS))]
ID_MAP = dict(zip(BDG_IDS, BLDS))
print(f"Selected site: {SITE} · buildings:", ID_MAP)

IDX = pd.date_range(_e.index.min(), _e.index.max(), freq="h")


def _clean(df):
    df = df.reindex(IDX)[BDG_IDS].apply(pd.to_numeric, errors="coerce").clip(lower=0)
    df = df.interpolate(limit=6, limit_area="inside")
    df.columns = [ID_MAP[c] for c in df.columns]
    return df


ELEC_H = _clean(_e)
WATER_H = _clean(_w)
del _e, _w

if P_WX:
    _wx = pd.read_csv(P_WX, parse_dates=["timestamp"])
    _wx = _wx[_wx.site_id == SITE].drop_duplicates("timestamp").set_index("timestamp")
    TEMP_H = _wx["airTemperature"].reindex(IDX).interpolate(limit=24).ffill().bfill()
else:
    TEMP_H = pd.Series(15.0, index=IDX)   # no weather → model falls back on calendar features
TEMP_D = TEMP_H.resample("D").mean()

BUILDINGS = pd.DataFrame({
    "building": BLDS, "bdg2_id": BDG_IDS,
    "use": picked.sub_primaryspaceusage.fillna("Education").values,
    "area_sqm": picked.sqm.round(0).values,
    "timezone": picked.timezone.values,
})
print(BUILDINGS.to_string(index=False))
print(f"Hourly range: {IDX.min()} → {IDX.max()}  ({len(IDX):,} hours)")
print("Electricity coverage:", ELEC_H.notna().mean().round(3).to_dict())
print("Water coverage:", WATER_H.notna().mean().round(3).to_dict())

## 3 · Solid-waste stream (synthetic, labelled as such)

In [ ]:
# ============================================================
# 3 · SOLID WASTE STREAM (SYNTHETIC — clearly labelled)
# BDG2 has no waste data. We generate daily wet / dry / recyclable kg per
# building from an occupancy proxy (each building's own weekday/term pattern
# in electricity), so waste rises and falls with real campus activity.
# ============================================================
_rng = np.random.default_rng(SEED)
_ed_clean = ELEC_H.resample("D").sum(min_count=20)
DAYS = _ed_clean.index

WASTE_D = {}           # building -> DataFrame[wet, dry, recyclable] (kg/day)
for b in BLDS:
    e = _ed_clean[b].interpolate(limit_area="inside").bfill().ffill()
    lo, hi = e.quantile(0.05), e.quantile(0.95)
    activity = ((e - lo) / max(hi - lo, 1e-6)).clip(0.05, 1.0)          # 0 = empty campus, 1 = peak term
    area = float(BUILDINGS.set_index("building").loc[b, "area_sqm"])
    total = area * 0.006 * (0.25 + 0.75 * activity) * _rng.lognormal(0, 0.10, len(DAYS))
    shares = _rng.dirichlet([45, 35, 20], len(DAYS))                       # wet / dry / recyclable
    WASTE_D[b] = pd.DataFrame({"wet": total * shares[:, 0], "dry": total * shares[:, 1],
                               "recyclable": total * shares[:, 2]}, index=DAYS).round(1)

print("Synthetic waste, mean kg/day per building:",
      {b: round(WASTE_D[b].sum(axis=1).mean(), 1) for b in BLDS})

## 4 · Verification engine: comparison-group counterfactual + placebo-in-time intervals
The core of *"without overstating environmental benefits"*.

In [ ]:
# ============================================================
# 4 · VERIFICATION ENGINE — comparison-group counterfactual + placebo-in-time CIs
#
# Question it answers: "Did this action really save anything, and how sure are we?"
#  1. Baseline model trained on the 365 days BEFORE the action. Inputs: day-of-week,
#     temperature, and the same-day readings of COMPARISON buildings that had no
#     action (they share the campus calendar, holidays, weather and occupancy swings).
#  2. Predict what this building WOULD have used after the action (counterfactual).
#  3. Raw effect = (counterfactual − actual) / counterfactual.
#  4. Repeat at many FAKE action dates before the real one (placebos). Nothing
#     happened then, so their "effects" measure how much saving the method
#     invents from noise alone.
#  5. Estimate = raw − bias (mean of the 8 most recent placebos);
#     95% prediction interval from the placebo spread.
#  6. VERIFIED only if the whole interval is above zero AND two safeguards hold
#     (≥2 comparison buildings; placebo dates span ≥150 days). Otherwise: NOT PROVEN.
#  Days flagged by the anomaly detector (leaks, faults) are excluded from every
#  window — they are non-routine events, not evidence about the action.
# ============================================================
MV_RECENT_PLACEBOS = 8
MV_MIN_TRAIN_DAYS = 200       # placebo windows may use 200–365 days of training, so the null reaches back far
                              # enough to include a previous term start / holiday transition
MV_CI_INFLATE = {"electricity": 1.2, "water": 1.0, "waste": 1.0}  # interval calibration, fitted on a SEPARATE
                              # set of 60 simulated actions (seed SEED+101); never shrinks an interval
MV_MIN_CONTROLS = 2           # safeguard: need ≥2 untreated comparison buildings to certify
MV_MIN_PLACEBO_SPAN = 150     # safeguard: placebo dates must span ≥150 days (seasonal behaviour represented)
MV_CONTROL_EXCLUDE_DAYS = 90  # a comparison building is unusable if it had its own action ≤90 days before / during
ACTION_REGISTRY = []      # filled by the action register; used to exclude contaminated controls
NONROUTINE_DAYS = {}      # (resource, building) -> set of days with an alert (excluded, IPMVP non-routine events)


def daily_series(resource, b, observed=True):
    if resource == "waste":
        return (OBS_WASTE_D if observed else WASTE_D)[b]["dry"].copy()
    h = (HOURLY_OBS if observed else HOURLY_CLEAN)[resource][b]
    return h.resample("D").sum(min_count=20)


def daily_feats(dix):
    """Calendar + weather features (also used by the waste anomaly model)."""
    f = pd.DataFrame(index=dix)
    f["dow"] = dix.dayofweek
    f["wkend"] = (f["dow"] >= 5).astype(int)
    doy = dix.dayofyear
    f["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    f["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    t = TEMP_D.reindex(dix).values
    f["temp"] = t
    f["hdd"] = np.clip(15.5 - t, 0, None)
    f["cdd"] = np.clip(t - 20.0, 0, None)
    return f


def comparison_buildings(resource, b, T, post_days, registry=None):
    """Other buildings with no logged action on this resource shortly before or during the window."""
    registry = ACTION_REGISTRY if registry is None else registry
    lo, hi = T - pd.Timedelta(days=MV_CONTROL_EXCLUDE_DAYS), T + pd.Timedelta(days=post_days)
    busy = {a["building"] for a in registry
            if a["resource"] == resource and lo <= pd.Timestamp(a["date"]) <= hi}
    return [o for o in BLDS if o != b and o not in busy]


def mv_design(resource, b, controls, observed=True):
    dix = DAYS
    X = pd.DataFrame(index=dix)
    X["dow"] = dix.dayofweek
    X["wkend"] = (X["dow"] >= 5).astype(int)
    X["temp"] = TEMP_D.reindex(dix).values
    if controls:
        for o in controls:
            X[f"ctrl_{o}"] = daily_series(resource, o, observed).reindex(dix).values
    else:                                    # no clean comparison group → calendar fallback
        f = daily_feats(dix)
        X[["doy_sin", "doy_cos", "hdd", "cdd"]] = f[["doy_sin", "doy_cos", "hdd", "cdd"]]
    return X


def _fit_window(y, X, T, post_days, train_days=MV_TRAIN_DAYS):
    tr = y.loc[T - pd.Timedelta(days=train_days): T - pd.Timedelta(days=1)].dropna()
    te = y.loc[T: T + pd.Timedelta(days=post_days - 1)].dropna()
    if len(tr) < 0.85 * MV_MIN_TRAIN_DAYS or len(te) < 0.6 * post_days:
        return None
    mdl = lgb.LGBMRegressor(**LGB_DAILY).fit(X.loc[tr.index], tr.values)
    pred = pd.Series(mdl.predict(X.loc[te.index]), index=te.index).clip(lower=0)
    eff = (pred.sum() - te.sum()) / max(pred.sum(), 1e-9) * 100
    return {"eff": float(eff), "pred": pred, "act": te}


def verify_action(resource, building, T, post_days=MV_POST_DAYS_DEFAULT, step=MV_PLACEBO_STEP,
                  observed=True, y=None, registry=None, min_placebos=8):
    """Placebo-in-time verification of one action. `y` overrides the treated series (validation)."""
    T = pd.Timestamp(T).normalize()
    y = (daily_series(resource, building, observed) if y is None else y).copy()
    start, end = y.dropna().index.min(), y.dropna().index.max()
    if T + pd.Timedelta(days=post_days - 1) > end:
        have = max(0, (end - T).days + 1)
        return {"verdict": "AWAITING DATA",
                "why": f"Needs {post_days} days of readings after the action; {have} available so far."}
    controls = comparison_buildings(resource, building, T, post_days, registry)
    X = mv_design(resource, building, controls, observed)
    excluded = 0
    if observed:
        own = y.index.intersection(pd.DatetimeIndex(sorted(NONROUTINE_DAYS.get((resource, building), set()))))
        excluded = len(own)
        y.loc[own] = np.nan                                   # drop the building's own abnormal days
        X = X.copy()
        for o in controls:                                    # blank (not drop) abnormal control readings
            bad = X.index.intersection(pd.DatetimeIndex(sorted(NONROUTINE_DAYS.get((resource, o), set()))))
            X.loc[bad, f"ctrl_{o}"] = np.nan
    real = _fit_window(y, X, T, post_days)
    if real is None:
        return {"verdict": "INSUFFICIENT HISTORY", "why": "Not enough clean readings before/after the action."}
    null, pdates, Tk = [], [], T - pd.Timedelta(days=post_days)
    while Tk - pd.Timedelta(days=MV_MIN_TRAIN_DAYS) >= start:
        r = _fit_window(y, X, Tk, post_days)
        if r is not None:
            null.append(r["eff"]); pdates.append(Tk)
        Tk -= pd.Timedelta(days=step)
    null = np.array(null)                      # ordered: most recent placebo first
    K = len(null)
    if K < min_placebos:
        return {"verdict": "INSUFFICIENT HISTORY",
                "why": f"Only {K} placebo windows available (need ≥{min_placebos}); ~15 months of history required."}
    nb = min(MV_RECENT_PLACEBOS, K)
    bias = float(null[:nb].mean())
    spread = float(np.sqrt(null.var(ddof=1) + null[:nb].var(ddof=1) / nb))
    est = real["eff"] - bias
    half = float(stats.t.ppf(0.975, K - 1) * spread * np.sqrt(1 + 1 / K)) * MV_CI_INFLATE.get(resource, 1.0)
    lo, hi = est - half, est + half
    p_val = float((1 + np.sum(np.abs(null - bias) >= abs(est))) / (K + 1))
    verdict = "VERIFIED SAVING" if lo > 0 else ("VERIFIED INCREASE" if hi < 0 else "NOT PROVEN")
    span = int((pdates[0] - pdates[-1]).days) if K > 1 else 0
    guard = []
    if len(controls) < MV_MIN_CONTROLS:
        guard.append(f"only {len(controls)} untreated comparison building(s) available (need ≥{MV_MIN_CONTROLS})")
    if span < MV_MIN_PLACEBO_SPAN:
        guard.append(f"placebo history spans only {span} days (need ≥{MV_MIN_PLACEBO_SPAN} to rule out seasonal effects)")
    if guard and verdict != "NOT PROVEN":
        verdict = "NOT PROVEN"
    pre = y.loc[T - pd.Timedelta(days=28): T - pd.Timedelta(days=1)].mean()
    naive = float((pre - real["act"].mean()) / pre * 100) if pre and pre > 0 else float("nan")
    act_total = float(real["act"].sum())
    cf_total = act_total / (1 - est / 100) if est < 99 else float(real["pred"].sum())
    scale = cf_total / max(float(real["pred"].sum()), 1e-9)
    return {
        "verdict": verdict, "estimate_pct": est, "ci_lo_pct": lo, "ci_hi_pct": hi,
        "raw_pct": real["eff"], "placebo_bias_pct": bias, "p_value": p_val,
        "naive_pct": naive, "n_placebos": K, "null": (null - bias).tolist(),
        "post_days": post_days, "action_date": T, "controls": controls, "excluded_days": excluded,
        "placebo_span_days": span, "guard": guard,
        "actual_total": act_total, "counterfactual_total": cf_total,
        "saved": cf_total * est / 100, "saved_lo": cf_total * lo / 100, "saved_hi": cf_total * hi / 100,
        "pred": real["pred"] * scale, "act": real["act"], "why": "",
    }


def explain_verdict(r, unit, label):
    if r["verdict"] in ("AWAITING DATA", "INSUFFICIENT HISTORY"):
        return r["why"]
    est, lo, hi, naive = r["estimate_pct"], r["ci_lo_pct"], r["ci_hi_pct"], r["naive_pct"]
    ctrl = (f"compared against {len(r['controls'])} untreated building(s) ({', '.join(r['controls'])})"
            if r["controls"] else "compared against a calendar-and-weather baseline (no untreated buildings available)")
    if r["verdict"] == "VERIFIED SAVING":
        return (f"{label} was {est:.1f}% below the counterfactual ({ctrl}); 95% interval {lo:.1f}% to {hi:.1f}%. "
                f"The whole interval is above zero, so the saving is real. Conservative claim: at least "
                f"{r['saved_lo']:,.0f} {unit} over {r['post_days']} days (we report the lower bound).")
    if r["verdict"] == "VERIFIED INCREASE":
        return (f"{label} was {abs(est):.1f}% ABOVE the counterfactual ({ctrl}); 95% interval {lo:.1f}% to {hi:.1f}%. "
                f"This action did not save resources — investigate.")
    if r.get("guard") and (lo > 0 or hi < 0):
        return (f"The interval ({lo:.1f}% to {hi:.1f}%, {ctrl}) excludes zero, but we do NOT certify it because "
                f"{' and '.join(r['guard'])}. Savings NOT confirmed, so none are claimed — collect more data or "
                f"keep an untreated building as a comparison group.")
    msg = (f"Estimated change {est:+.1f}% ({ctrl}), but the 95% interval ({lo:.1f}% to {hi:.1f}%) includes zero: "
           f"normal variation here is that large. Savings NOT confirmed, so none are claimed.")
    if r.get("guard"):
        msg += f" (Also: {' and '.join(r['guard'])}.)"
    if not np.isnan(naive) and naive > hi + 2:
        msg += (f" A simple before/after comparison would have claimed {naive:.1f}% — that drop is explained by "
                f"the campus calendar/season (the untreated buildings dropped too), not by this action.")
    return msg


print("Verification engine ready.")

## 5 · Labelled sample dataset
Real readings + injected interventions (known true effects, including one with **zero** effect) + injected anomalies (known timing).

In [ ]:
# ============================================================
# 5 · LABELLED SAMPLE DATASET (required output)
# Real readings + INJECTED interventions with KNOWN true effects +
# INJECTED anomalies with KNOWN timing. Because we know the truth, we can
# measure how well the platform detects problems and verifies savings.
# ============================================================
_rng = np.random.default_rng(SEED + 1)
OBS_ELEC_H, OBS_WATER_H = ELEC_H.copy(), WATER_H.copy()
OBS_WASTE_D = {b: WASTE_D[b].copy() for b in BLDS}
HOURLY_OBS = {"electricity": OBS_ELEC_H, "water": OBS_WATER_H}
HOURLY_CLEAN = {"electricity": ELEC_H, "water": WATER_H}


DATA_END = IDX.max().normalize()
LATEST_ACTION = DATA_END - pd.Timedelta(days=MV_POST_DAYS_DEFAULT)

# ---------- choose scenario buildings (documented, data-driven) ----------
_mean_e = ELEC_H.loc[:TRAIN_END].mean().sort_values(ascending=False)
B_LED = _mean_e.index[0]                                   # biggest electricity user gets the LED retrofit
_meter_err = {}
for b in BLDS:   # most reliable water meter gets the tap retrofit: smallest |error| + interval on CLEAN data
    r = verify_action("water", b, pd.Timestamp("2017-08-21"), step=10, observed=False, registry=[])
    _meter_err[b] = abs(r["estimate_pct"]) + (r["ci_hi_pct"] - r["ci_lo_pct"]) / 2 if "estimate_pct" in r else 1e9
B_TAPS = min(_meter_err, key=_meter_err.get)
print("Water-meter reliability (lower = better):", {k: round(v, 1) for k, v in _meter_err.items()})
B_WASTE = max(BLDS, key=lambda b: WASTE_D[b]["dry"].mean())
_rest = [b for b in BLDS if b != B_LED]
B_HVAC = _rest[0]

# Awareness campaign with ZERO true effect, dated where the WHOLE CAMPUS drops (term end / season),
# so a naive before/after looks impressive while the drop is really campus-wide.
_cands = [x for x in _rest if x != B_HVAC]
best = (None, None, -1e9, 0.0)
_eser = {x: daily_series("electricity", x, False) for x in BLDS}
for T in pd.date_range("2017-04-15", LATEST_ACTION, freq="D"):
    drops = {}
    for x in BLDS:
        s_ = _eser[x]
        pre = s_.loc[T - pd.Timedelta(days=28): T - pd.Timedelta(days=1)].mean()
        post = s_.loc[T: T + pd.Timedelta(days=MV_POST_DAYS_DEFAULT - 1)].mean()
        drops[x] = (pre - post) / pre * 100
    med = float(np.median(list(drops.values())))
    if med > best[2]:
        typical = min(_cands, key=lambda x: abs(drops[x] - med))
        best = (typical, T, med, drops[typical])
B_CAMPAIGN, T_CAMPAIGN, _campus_drop, _naive_drop = best

ACTIONS_TRUTH = [
    # id, title, building, resource, date, cost ₹, claimed %, how the truth is injected
    dict(action_id="A1", title="LED lighting retrofit (all floors)", building=B_LED, resource="electricity",
         date=pd.Timestamp("2017-09-04"), cost_inr=1_850_000, claimed_pct=15.0,
         inject=dict(kind="hours", hours=list(range(7, 22)), pct=20.0)),
    dict(action_id="A2", title="Sensor taps + dual-flush retrofit", building=B_TAPS, resource="water",
         date=pd.Timestamp("2017-08-21"), cost_inr=420_000, claimed_pct=25.0,
         inject=dict(kind="all", pct=22.0)),
    dict(action_id="A3", title="'Switch-off' awareness campaign", building=B_CAMPAIGN, resource="electricity",
         date=T_CAMPAIGN, cost_inr=60_000, claimed_pct=20.0,
         inject=dict(kind="none", pct=0.0)),
    dict(action_id="A4", title="HVAC night set-back schedule", building=B_HVAC, resource="electricity",
         date=pd.Timestamp("2017-10-02"), cost_inr=35_000, claimed_pct=8.0,
         inject=dict(kind="hours", hours=[20, 21, 22, 23, 0, 1, 2, 3, 4, 5], pct=8.0)),
    dict(action_id="A5", title="Source-segregation drive (dry → recyclable)", building=B_WASTE, resource="waste",
         date=pd.Timestamp("2017-09-11"), cost_inr=90_000, claimed_pct=30.0,
         inject=dict(kind="divert", pct=28.0)),
]

ACTION_REGISTRY[:] = [dict(building=a["building"], resource=a["resource"], date=a["date"]) for a in ACTIONS_TRUTH]

for a in ACTIONS_TRUTH:
    T, b, inj = a["date"], a["building"], a["inject"]
    if a["resource"] in HOURLY_OBS:
        H = HOURLY_OBS[a["resource"]]
        m = H.index >= T
        if inj["kind"] == "hours":
            m &= np.isin(H.index.hour, inj["hours"])
        if inj["kind"] != "none":
            H.loc[m, b] = H.loc[m, b] * (1 - inj["pct"] / 100)
    else:
        W = OBS_WASTE_D[b]
        m = W.index >= T
        moved = W.loc[m, "dry"] * inj["pct"] / 100
        W.loc[m, "dry"] -= moved
        W.loc[m, "recyclable"] += moved


def true_effect_pct(a, post_days=MV_POST_DAYS_DEFAULT):
    """Exact injected effect over the measurement window (ground truth)."""
    T = pd.Timestamp(a["date"])
    c = daily_series(a["resource"], a["building"], False).loc[T: T + pd.Timedelta(days=post_days - 1)]
    o = daily_series(a["resource"], a["building"], True).loc[T: T + pd.Timedelta(days=post_days - 1)]
    return float((c.sum() - o.sum()) / c.sum() * 100)


# ---------- anomalies with known timing ----------
ANOMALY_SPEC = [("electricity", "equipment_left_on")] * 2 + [("electricity", "weekend_hvac_override")] * 2 + \
               [("electricity", "gradual_drift")] * 2 + [("water", "leak")] * 3 + [("water", "stuck_valve")] * 2 + \
               [("waste", "dumping_spike")] * 2
ANOMALY_SPEC = ANOMALY_SPEC * 2          # 26 held-out anomalies


def inject_anomalies(obs_hourly, obs_waste, rng, spec, actions=(), prefix="X"):
    """Add anomalies of known type/timing to the observed series (in place). Returns truth rows."""
    truth = []
    win_s, win_e = EVAL_START + pd.Timedelta(days=3), DATA_END - pd.Timedelta(days=12)

    def busy(b, res, s, e):
        for a in actions:
            if a["building"] == b and a["resource"] == res and \
               s < a["date"] + pd.Timedelta(days=MV_POST_DAYS_DEFAULT + 7) and e > a["date"] - pd.Timedelta(days=3):
                return True
        for x in truth:
            if x["building"] == b and x["resource"] == res and \
               s < x["end"] + pd.Timedelta(days=4) and e > x["start"] - pd.Timedelta(days=4):
                return True
        return False

    for n, (res, kind) in enumerate(spec, 1):
        for _try in range(1000):
            b = BLDS[rng.integers(len(BLDS))]
            day = win_s + pd.Timedelta(days=int(rng.integers(0, (win_e - win_s).days)))
            if kind == "equipment_left_on":
                s = day + pd.Timedelta(hours=int(rng.integers(19, 23))); e = s + pd.Timedelta(hours=int(rng.integers(8, 14)))
            elif kind == "weekend_hvac_override":
                s = day - pd.Timedelta(days=(day.dayofweek - 5) % 7); e = s + pd.Timedelta(hours=47)
            elif kind == "gradual_drift":
                s = day; e = s + pd.Timedelta(days=int(rng.integers(10, 15)))
            elif kind == "leak":
                s = day + pd.Timedelta(hours=int(rng.integers(0, 24))); e = s + pd.Timedelta(days=int(rng.integers(3, 7)))
            elif kind == "stuck_valve":
                s = day + pd.Timedelta(hours=int(rng.integers(7, 12))); e = s + pd.Timedelta(hours=int(rng.integers(8, 14)))
            else:
                s = day; e = s + pd.Timedelta(days=int(rng.integers(1, 3)))
            if s >= win_s and e <= DATA_END and not busy(b, res, s, e):
                break
        if res in obs_hourly:
            H = obs_hourly[res]
            base = float(HOURLY_CLEAN[res].loc[:TRAIN_END, b].median())
            ix = H.index[(H.index >= s) & (H.index < e)]
            if kind == "equipment_left_on":
                add = np.full(len(ix), base * rng.uniform(0.6, 0.9))
            elif kind == "weekend_hvac_override":
                add = np.full(len(ix), base * rng.uniform(0.4, 0.6))
            elif kind == "gradual_drift":
                add = base * rng.uniform(0.30, 0.40) * np.linspace(0, 1, len(ix))
            elif kind == "leak":
                add = np.full(len(ix), base * rng.uniform(0.6, 1.0))
            else:   # stuck valve: sized against normal DAYTIME peak flow
                hi = float(HOURLY_CLEAN[res].loc[:TRAIN_END, b].quantile(0.9))
                add = np.full(len(ix), hi * rng.uniform(1.0, 1.5))
            H.loc[ix, b] = H.loc[ix, b] + add
            excess = float(np.sum(add))
        else:
            W = obs_waste[b]
            d_ix = W.index[(W.index >= s.normalize()) & (W.index < e)]
            add = pd.Series(float(WASTE_D[b]["dry"].loc[:TRAIN_END].mean()) * rng.uniform(0.8, 1.5), index=d_ix)
            W.loc[d_ix, "dry"] += add
            excess = float(add.sum())
        truth.append(dict(anomaly_id=f"{prefix}{n:02d}", building=b, resource=res, kind=kind,
                          start=s, end=e, injected_excess=round(excess, 1)))
    return truth


ANOMALY_TRUTH = inject_anomalies(HOURLY_OBS, OBS_WASTE_D, _rng, ANOMALY_SPEC, ACTIONS_TRUTH)

ACTIONS_TRUTH_DF = pd.DataFrame([{k: v for k, v in a.items() if k != "inject"} | {
    "injected_true_effect_pct": round(true_effect_pct(a), 2),
    "injection": json.dumps(a["inject"])} for a in ACTIONS_TRUTH])
ANOMALY_TRUTH_DF = pd.DataFrame(ANOMALY_TRUTH)


def export_labelled_dataset():
    """Write the labelled sample dataset (CSV files in one ZIP)."""
    path = os.path.join(OUT_DIR, "greenledger_labelled_dataset.zip")
    rows = []
    for res, H in HOURLY_OBS.items():
        lab = pd.DataFrame("", index=H.index, columns=H.columns)
        for x in ANOMALY_TRUTH:
            if x["resource"] == res:
                lab.loc[(lab.index >= x["start"]) & (lab.index < x["end"]), x["building"]] = x["anomaly_id"]
        long = H.stack(future_stack=True).rename("observed").to_frame()
        long["clean_real"] = HOURLY_CLEAN[res].stack(future_stack=True)
        long["anomaly_id"] = lab.stack(future_stack=True)
        long["resource"] = res
        rows.append(long)
    hourly = pd.concat(rows).reset_index()
    hourly.columns = ["timestamp", "building", "observed", "clean_real", "anomaly_id", "resource"]
    waste = pd.concat({b: OBS_WASTE_D[b] for b in BLDS}, names=["building", "date"]).reset_index()
    readme = (
        "GreenLedger labelled sample dataset\n"
        f"Electricity (kWh) & water (litres): REAL hourly readings, Building Data Genome 2, site '{SITE}'.\n"
        "Weather: real (BDG2 weather.csv). Solid waste (kg/day): SYNTHETIC, driven by an occupancy proxy.\n"
        "'observed' = what the platform sees; 'clean_real' = original reading before injection.\n"
        "anomalies_truth.csv: injected problems (known start/end). actions_truth.csv: injected interventions\n"
        "with the exact true effect, including one action (A3) whose true effect is ZERO.\n")
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("README.txt", readme)
        z.writestr("hourly_electricity_water.csv", hourly.to_csv(index=False))
        z.writestr("daily_waste.csv", waste.to_csv(index=False))
        z.writestr("anomalies_truth.csv", ANOMALY_TRUTH_DF.to_csv(index=False))
        z.writestr("actions_truth.csv", ACTIONS_TRUTH_DF.to_csv(index=False))
        z.writestr("buildings.csv", BUILDINGS.to_csv(index=False))
    return path


LABELLED_ZIP = export_labelled_dataset()
print(ACTIONS_TRUTH_DF[["action_id", "title", "building", "resource", "date", "claimed_pct", "injected_true_effect_pct"]].to_string(index=False))
print(f"\nZero-effect campaign: {B_CAMPAIGN} on {T_CAMPAIGN:%d %b %Y} — campus-wide drop {_campus_drop:.1f}%, "
      f"this building {_naive_drop:.1f}%")
print(ANOMALY_TRUTH_DF.to_string(index=False))
print("Labelled dataset:", LABELLED_ZIP)

## 6 · Unusual-change detection

In [ ]:
# ============================================================
# 6 · UNUSUAL-CHANGE DETECTION
# Expected-consumption model per building & resource (calendar + weather),
# trained only on data before the monitoring period. Residuals are scaled by
# OUT-OF-SAMPLE error (time-blocked CV) so alerts are not overconfident, and
# de-biased against a lagged 4-week median so slow seasonal/regime shifts do
# not trigger alerts. Only EXCESS consumption is alerted.
#   Electricity : hourly spike rule  +  CUSUM on daily excess (drifts)
#   Water       : Minimum Night Flow (01:00–05:00) — standard leak indicator
#                 +  6-hour sustained excess (stuck valve / overflow) — natural
#                    one-hour spikes in real water data are ignored
#   Waste       : abnormal daily dry-waste volume
# Thresholds were tuned on a SEPARATE development set of randomly injected
# anomalies (cell 6b), never on the labelled evaluation set.
# ============================================================
DET = dict(z_hour=3.0, z_peak=7.0, min_hours=5, merge_gap_h=3,      # frozen after dev-set tuning (cell 6b)
           cusum_k=1.5, cusum_h=6.0,
           mnf_k=0.75, mnf_h=6.0, mnf_z_single=6.0,
           water_h6_z=8.0, waste_day_z=3.0)
NIGHT_HOURS = [1, 2, 3, 4]


def hourly_feats(ix):
    f = pd.DataFrame(index=ix)
    f["hour"] = ix.hour
    f["dow"] = ix.dayofweek
    f["how"] = f["dow"] * 24 + f["hour"]
    f["wkend"] = (f["dow"] >= 5).astype(int)
    doy = ix.dayofyear
    f["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    f["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    t = TEMP_H.reindex(ix)
    f["temp"] = t.values
    f["temp24"] = t.rolling(24, min_periods=1).mean().values
    return f


def _robust_sd(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    return max(1.4826 * np.median(np.abs(x - np.median(x))), 1e-6)


def _cv_residuals(y, featfn, params, n_folds=4):
    """Out-of-sample residuals over the training period via time-blocked folds."""
    y = y.dropna()
    res = pd.Series(np.nan, index=y.index)
    X = featfn(y.index)
    for f in np.array_split(np.arange(len(y)), n_folds):
        mask = np.ones(len(y), bool); mask[f] = False
        m = lgb.LGBMRegressor(**params).fit(X.iloc[mask], y.iloc[mask])
        res.iloc[f] = y.iloc[f].values - m.predict(X.iloc[f])
    return res


def _debias(r, window, lag):
    """Remove slow level shifts: subtract median residual of an earlier window."""
    base = r.rolling(window, min_periods=window // 3).median().shift(lag)
    return r - base.fillna(0)


def _debias_hourly(r, days=28, lag_days=7):
    """Per-hour-of-day de-bias, so a daytime-only change (e.g. a lighting retrofit)
    does not shift the night-time baseline."""
    base = r.groupby(r.index.hour).transform(
        lambda s: s.rolling(days, min_periods=days // 3).median().shift(lag_days))
    return r - base.fillna(0)


def _night_min(s):
    n = s[np.isin(s.index.hour, NIGHT_HOURS)]
    return n.resample("D").min()


# ---------- train expected-consumption models (once) ----------
EXPECTED, SIG = {}, {}
MODEL_FIT = []
t0 = time.time()
for res in ("electricity", "water"):
    for b in BLDS:
        y_tr = HOURLY_CLEAN[res][b].loc[:TRAIN_END].dropna()
        cvr = _cv_residuals(y_tr, hourly_feats, LGB_HOURLY)
        mdl = lgb.LGBMRegressor(**LGB_HOURLY).fit(hourly_feats(y_tr.index), y_tr.values)
        exp = pd.Series(mdl.predict(hourly_feats(IDX)), index=IDX).clip(lower=0)
        EXPECTED[(res, b)] = exp
        cvr_db = _debias_hourly(cvr)
        day_db = _debias(cvr.resample("D").sum(min_count=20), 28, 7)
        key = (cvr_db.index.dayofweek >= 5).astype(int) * 24 + cvr_db.index.hour     # weekend × hour buckets
        sig = {"hour": cvr_db.groupby(key).apply(_robust_sd).reindex(range(48)).ffill().bfill(),
               "day": _robust_sd(day_db)}
        if res == "water":
            exp_cv = y_tr - cvr
            mnf_r = _night_min(y_tr) - _night_min(exp_cv)
            sig["mnf"] = _robust_sd(_debias(mnf_r, 28, 7))
            sig["h6"] = _robust_sd(_debias_hourly(cvr).rolling(6).sum())   # 6-hour sustained excess
        SIG[(res, b)] = sig
        # out-of-sample accuracy of the baseline AS USED (model + recent-normal adjustment) on CLEAN real data
        full = HOURLY_CLEAN[res][b]
        adj = (full - _debias_hourly(full - exp)).loc[EVAL_START:]
        clean = full.loc[EVAL_START:]
        ok = clean.notna() & adj.notna(); mean_ = clean[ok].mean()
        cd, ad = clean.resample("D").sum(min_count=20), adj.resample("D").sum()
        okd = cd.notna()
        MODEL_FIT.append(dict(resource=res, building=b,
                              cv_rmse_hourly_pct=round(np.sqrt(np.mean((clean[ok] - adj[ok]) ** 2)) / mean_ * 100, 1),
                              nmbe_pct=round((adj[ok] - clean[ok]).sum() / (ok.sum() * mean_) * 100, 1),
                              cv_rmse_daily_pct=round(np.sqrt(np.mean((cd[okd] - ad[okd]) ** 2)) / cd[okd].mean() * 100, 1)))

for b in BLDS:
    y_tr = WASTE_D[b]["dry"].loc[:TRAIN_END]
    cvr = _cv_residuals(y_tr, daily_feats, LGB_DAILY)
    mdl = lgb.LGBMRegressor(**LGB_DAILY).fit(daily_feats(y_tr.index), y_tr.values)
    exp = pd.Series(mdl.predict(daily_feats(DAYS)), index=DAYS).clip(lower=0)
    EXPECTED[("waste", b)] = exp
    SIG[("waste", b)] = {"day": _robust_sd(_debias(cvr, 28, 7))}
    full = WASTE_D[b]["dry"]
    adj = (full - _debias(full - exp, 28, 7)).loc[EVAL_START:]
    clean = full.loc[EVAL_START:]
    MODEL_FIT.append(dict(resource="waste", building=b, cv_rmse_hourly_pct=np.nan,
                          nmbe_pct=round((adj - clean).sum() / (len(clean) * clean.mean()) * 100, 1),
                          cv_rmse_daily_pct=round(np.sqrt(np.mean((clean - adj) ** 2)) / clean.mean() * 100, 1)))
MODEL_FIT_DF = pd.DataFrame(MODEL_FIT)
print(f"Expected-consumption models trained in {time.time() - t0:.0f}s")


# ---------- detection ----------
def _segments(ts, gap):
    segs, cur = [], []
    for t in ts:
        if cur and (t - cur[-1]) > gap:
            segs.append(cur); cur = []
        cur.append(t)
    if cur:
        segs.append(cur)
    return segs


def _cusum(zd, k, h):
    out, S, s0 = [], 0.0, None
    for d, v in zd.items():
        S_new = max(0.0, S + v - k)
        if S == 0 and S_new > 0:
            s0 = d
        if S_new >= h and S < h and not (out and out[-1][1] is None):
            out.append([s0, None, d + pd.Timedelta(days=1)])
        if S_new == 0 and out and out[-1][1] is None:
            out[-1][1] = d
        S = S_new
    if out and out[-1][1] is None:
        out[-1][1] = zd.index[-1] + pd.Timedelta(days=1)
    return out


def _guess_kind(res, s, e, rule, night_share, wk_share):
    dur_h = (e - s) / pd.Timedelta(hours=1)
    if res == "water":
        return ("Night-time flow well above normal — likely leak" if rule == "mnf"
                else "Hours of sustained extra flow — stuck valve, overflow or burst")
    if res == "waste":
        return "Unusual dry-waste volume — event, dumping or missed segregation"
    if dur_h >= 72:
        return "Sustained rise over days — possible equipment fault / drift"
    if wk_share >= 0.6:
        return "Weekend load — HVAC or schedule override"
    if night_share >= 0.5:
        return "Night-time load — equipment or lights left on"
    return "Load spike above expected"


def detect_series(res, b, obs, p=None):
    """Return candidate alert dicts for one building/resource observed series."""
    p = p or DET
    exp, sig = EXPECTED[(res, b)], SIG[(res, b)]
    cands = []
    if res == "electricity":
        r = _debias_hourly(obs - exp).loc[EVAL_START:]
        z = r / sig["hour"].reindex((r.index.dayofweek >= 5).astype(int) * 24 + r.index.hour).values
        for seg in _segments(z.index[z >= p["z_hour"]], pd.Timedelta(hours=p["merge_gap_h"])):
            peak = z.loc[seg].max()
            if len(seg) >= p["min_hours"] or peak >= p["z_peak"]:
                hits = [t for t in seg if z.loc[t] >= p["z_peak"]]
                alarm = min(hits[0] if hits else seg[-1], seg[min(p["min_hours"], len(seg)) - 1])
                cands.append([seg[0], seg[-1] + pd.Timedelta(hours=1), alarm, "spike", float(peak)])
        rd = _debias((obs - exp).resample("D").sum(min_count=20), 28, 7).loc[EVAL_START:]
        zd = (rd / sig["day"]).fillna(0)
        for s, e, a in _cusum(zd, p["cusum_k"], p["cusum_h"]):
            cands.append([s, e, a, "cusum", float(zd.loc[s:e].max())])
    elif res == "water":
        mnf = _debias(_night_min(obs) - _night_min(exp), 28, 7).loc[EVAL_START:]
        zm = (mnf / sig["mnf"]).fillna(0)
        for s, e, a in _cusum(zm, p["mnf_k"], p["mnf_h"]):          # sustained night-flow rise
            cands.append([s, e, a - pd.Timedelta(hours=19), "mnf", float(zm.loc[s:e].max())])
        for d in zm.index[zm >= p["mnf_z_single"]]:                  # one extreme night
            cands.append([d, d + pd.Timedelta(days=1), d + pd.Timedelta(hours=5), "mnf", float(zm.loc[d])])
        r6 = _debias_hourly(obs - exp).rolling(6).sum().loc[EVAL_START:]
        z6 = (r6 / sig["h6"]).fillna(0)
        for seg in _segments(z6.index[z6 >= p["water_h6_z"]], pd.Timedelta(hours=p["merge_gap_h"])):
            cands.append([seg[0] - pd.Timedelta(hours=5), seg[-1] + pd.Timedelta(hours=1), seg[0], "h6",
                          float(z6.loc[seg].max())])
    else:
        rd = _debias(obs - exp, 28, 7).loc[EVAL_START:]
        zd = rd / sig["day"]
        for d in zd.index[zd >= p["waste_day_z"]]:
            cands.append([d, d + pd.Timedelta(days=1), d + pd.Timedelta(days=1), "day", float(zd.loc[d])])
    # merge overlapping candidates
    cands.sort(key=lambda c: c[0])
    merged = []
    for c in cands:
        if merged and c[0] <= merged[-1][1] + pd.Timedelta(hours=p["merge_gap_h"]):
            m = merged[-1]
            m[1] = max(m[1], c[1]); m[2] = min(m[2], c[2]); m[4] = max(m[4], c[4])
            if c[3] == "mnf": m[3] = "mnf"
        else:
            merged.append(list(c))
    out = []
    for s, e, alarm, rule, peak in merged:
        seg_o, seg_e = obs.loc[s: e - pd.Timedelta(seconds=1)], exp.loc[s: e - pd.Timedelta(seconds=1)]
        excess = float((seg_o - seg_e).clip(lower=0).sum())
        if excess <= 0:
            continue
        hrs = seg_o.index
        night = float(np.mean((hrs.hour >= 20) | (hrs.hour < 6))) if res != "waste" else 0
        wk = float(np.mean(hrs.dayofweek >= 5)) if res != "waste" else 0
        out.append(dict(building=b, resource=res, start=s, end=e, alarm_time=alarm, rule=rule,
                        peak_z=round(peak, 1), excess=round(excess, 1),
                        excess_pct=round(excess / max(seg_e.sum(), 1e-9) * 100, 1),
                        kind=_guess_kind(res, s, e, rule, night, wk)))
    return out


def detect_all(obs_hourly, obs_waste, p=None):
    ev = []
    for res in ("electricity", "water"):
        for b in BLDS:
            ev += detect_series(res, b, obs_hourly[res][b], p)
    for b in BLDS:
        ev += detect_series("waste", b, obs_waste[b], p)
    cols = ["building", "resource", "start", "end", "alarm_time", "rule", "peak_z", "excess", "excess_pct", "kind"]
    df = pd.DataFrame(ev, columns=cols).sort_values("alarm_time").reset_index(drop=True)
    df.insert(0, "alert_id", [f"AL-{i + 1:03d}" for i in range(len(df))])
    return df


def evaluate_detection(events, truth, tol_h=12):
    tol = pd.Timedelta(hours=tol_h)
    matched_ev, rows = set(), []
    for _, t in truth.iterrows():
        hit = events[(events.building == t.building) & (events.resource == t.resource)
                     & (events.start <= t.end + tol) & (events.end >= t.start - tol)]
        if len(hit):
            first = hit.sort_values("alarm_time").iloc[0]
            matched_ev.update(hit.alert_id)
            delay = max((first.alarm_time - t.start) / pd.Timedelta(hours=1), 0)
            rows.append(dict(anomaly_id=t.anomaly_id, building=t.building, resource=t.resource, kind=t.kind,
                             start=t.start, detected=True, alert_id=first.alert_id, delay_h=round(delay, 1)))
        else:
            rows.append(dict(anomaly_id=t.anomaly_id, building=t.building, resource=t.resource, kind=t.kind,
                             start=t.start, detected=False, alert_id="", delay_h=np.nan))
    per = pd.DataFrame(rows)
    tp = int(per.detected.sum())
    prec = len(matched_ev) / max(len(events), 1)
    rec = tp / max(len(truth), 1)
    months = max((IDX.max() - EVAL_START).days / 30.4, 1)
    summary = dict(injected=len(truth), detected=tp, recall=round(rec, 3),
                   alerts=len(events), alerts_on_injected=len(matched_ev), other_alerts=len(events) - len(matched_ev),
                   precision_lower_bound=round(prec, 3), f1=round(2 * prec * rec / max(prec + rec, 1e-9), 3),
                   other_alerts_per_series_month=round((len(events) - len(matched_ev)) / (3 * len(BLDS) * months), 2),
                   median_delay_h=round(float(per.delay_h.median()), 1) if tp else None)
    return summary, per


print("Detector ready.")

## 6b · Development set (threshold tuning only)

In [ ]:
# ============================================================
# 6b · DEVELOPMENT SET — thresholds were tuned here, on 48 randomly injected
# anomalies (different random seed, no interventions), then FROZEN.
# The labelled evaluation set (cell 5) was never used for tuning.
# ============================================================
DEV_SPEC = [("electricity", k) for k in ("equipment_left_on", "weekend_hvac_override", "gradual_drift")] * 7 + \
           [("water", k) for k in ("leak", "stuck_valve")] * 9 + [("waste", "dumping_spike")] * 9
DEV_HOURLY = {r: HOURLY_CLEAN[r].copy() for r in HOURLY_CLEAN}
DEV_WASTE = {b: WASTE_D[b].copy() for b in BLDS}
DEV_TRUTH = pd.DataFrame(inject_anomalies(DEV_HOURLY, DEV_WASTE, np.random.default_rng(SEED + 100), DEV_SPEC, prefix="D"))
DEV_WASTE_DRY = {b: DEV_WASTE[b]["dry"] for b in BLDS}
DEV_EVENTS = detect_all(DEV_HOURLY, DEV_WASTE_DRY)
DEV_SUMMARY, DEV_PER = evaluate_detection(DEV_EVENTS, DEV_TRUTH)
print("Development set (tuning only):", DEV_SUMMARY)
print(DEV_PER.groupby("kind").detected.mean().round(2).to_dict())

## 7 · Alerts on the held-out labelled set

In [ ]:
# ============================================================
# 7 · ALERTS ON THE LABELLED EVALUATION SET (held out — never used for tuning)
# ============================================================
EVENTS = detect_all(HOURLY_OBS, {b: OBS_WASTE_D[b]["dry"] for b in BLDS})
DETECT_SUMMARY, DETECT_PER = evaluate_detection(EVENTS, ANOMALY_TRUTH_DF)
_inj = DETECT_PER.set_index("alert_id").anomaly_id.to_dict() if len(DETECT_PER) else {}
print(f"Alerts raised in monitoring period ({EVAL_START.date()} → {DATA_END.date()}): {len(EVENTS)}")
print("Evaluation set:", DETECT_SUMMARY)
print(DETECT_PER.to_string(index=False))


def refresh_nonroutine(events):
    """Days touched by an alert become non-routine (excluded from savings verification)."""
    NONROUTINE_DAYS.clear()
    for _, e in events.iterrows():
        days = pd.date_range(pd.Timestamp(e.start).normalize(), (pd.Timestamp(e.end) - pd.Timedelta(seconds=1)).normalize(), freq="D")
        NONROUTINE_DAYS.setdefault((e.resource, e.building), set()).update(days)


refresh_nonroutine(EVENTS)
print("Non-routine days excluded from verification:", sum(len(v) for v in NONROUTINE_DAYS.values()))

## 8 · Verify every action, then validate the verifier

In [ ]:
# ============================================================
# 8 · VERIFY EVERY LOGGED ACTION + VALIDATE THE VERIFIER
# ============================================================
t0 = time.time()
MV_RESULTS = {}
for a in ACTIONS_TRUTH:
    MV_RESULTS[a["action_id"]] = verify_action(a["resource"], a["building"], a["date"])
print(f"Verified {len(MV_RESULTS)} actions in {time.time() - t0:.0f}s\n")
for a in ACTIONS_TRUTH:
    r = MV_RESULTS[a["action_id"]]
    if "estimate_pct" not in r:
        print(a["action_id"], r["verdict"], r["why"]); continue
    print(f'{a["action_id"]} {a["title"][:38]:38s} claimed {a["claimed_pct"]:5.1f}%  true {true_effect_pct(a):5.1f}%  '
          f'naive {r["naive_pct"]:6.1f}%  → {r["verdict"]:16s} {r["estimate_pct"]:6.1f}% [{r["ci_lo_pct"]:6.1f}, {r["ci_hi_pct"]:6.1f}]  K={r["n_placebos"]}')


def run_mv_validation(n_cases=45, seed=SEED + 7, post_days=MV_POST_DAYS_DEFAULT, step=7):
    """Monte-Carlo check on CLEAN real data: inject a known effect at a random date,
    run the verifier, and record whether the interval contains the truth."""
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n_cases):
        res = ["electricity", "water", "waste"][int(rng.integers(0, 3))]
        b = BLDS[int(rng.integers(len(BLDS)))]
        y = daily_series(res, b, False)
        # dates where ≥150 days of placebo history exist and the window fits in the data
        T = pd.Timestamp("2017-06-15") + pd.Timedelta(days=int(rng.integers(0, 150)))
        true = float(rng.choice([0, 0, 0, 5, 10, 15, 20, 25]))
        y2 = y.copy()
        post = y2.loc[T: T + pd.Timedelta(days=post_days - 1)].index
        y2.loc[post] = y2.loc[post] * (1 - true / 100)
        r = verify_action(res, b, T, post_days=post_days, step=step, observed=False, y=y2, registry=[])
        if "estimate_pct" not in r:
            continue
        rows.append(dict(resource=res, building=b, date=T.date(), true_pct=true, naive_pct=round(r["naive_pct"], 1),
                         estimate_pct=round(r["estimate_pct"], 1), ci_lo=round(r["ci_lo_pct"], 1),
                         ci_hi=round(r["ci_hi_pct"], 1), verdict=r["verdict"]))
    df = pd.DataFrame(rows)
    df["covered"] = (df.ci_lo <= df.true_pct) & (df.true_pct <= df.ci_hi)
    df["claimed_saving"] = df.verdict == "VERIFIED SAVING"
    zero = df[df.true_pct == 0]
    summary = dict(
        cases=len(df),
        ci_coverage=round(df.covered.mean(), 3),
        false_claims_on_zero_effect=f"{int(zero.claimed_saving.sum())} of {len(zero)}",
        false_claim_rate=round(zero.claimed_saving.mean(), 3) if len(zero) else None,
        naive_false_claims_on_zero_effect=f"{int((zero.naive_pct > 5).sum())} of {len(zero)} (naive >5%)",
        mean_abs_error_ours=round((df.estimate_pct - df.true_pct).abs().mean(), 2),
        mean_abs_error_naive=round((df.naive_pct - df.true_pct).abs().mean(), 2),
    )
    power = df[df.true_pct > 0].groupby("true_pct").claimed_saving.mean().round(2).to_dict()
    return summary, power, df


t0 = time.time()
MV_VALID_SUMMARY, MV_VALID_POWER, MV_VALID_DF = run_mv_validation()
print(f"\nVerifier validation ({time.time() - t0:.0f}s):", MV_VALID_SUMMARY)
print("Share verified, by true effect size:", MV_VALID_POWER)

## 9 · Platform store: action register, alert queue, decision log

In [ ]:
# ============================================================
# 9 · PLATFORM STORE (SQLite) — action register, alert queue, decision log
# ============================================================
for _sfx in ("", "-wal", "-shm"):
    if os.path.exists(DB_PATH + _sfx):
        os.remove(DB_PATH + _sfx)


def db():
    c = sqlite3.connect(DB_PATH, timeout=30, check_same_thread=False)
    c.row_factory = sqlite3.Row
    return c


with db() as c:
    c.executescript("""
    CREATE TABLE actions(action_id TEXT PRIMARY KEY, title TEXT, building TEXT, resource TEXT,
        date TEXT, cost_inr REAL, claimed_pct REAL, notes TEXT, source TEXT, created_at TEXT);
    CREATE TABLE verifications(action_id TEXT, post_days INT, as_of TEXT, verdict TEXT,
        estimate_pct REAL, ci_lo_pct REAL, ci_hi_pct REAL, naive_pct REAL, saved REAL, saved_lo REAL,
        saved_hi REAL, unit TEXT, n_placebos INT, controls TEXT, excluded_days INT, verified_at TEXT,
        PRIMARY KEY(action_id, post_days, as_of));
    CREATE TABLE alerts(alert_id TEXT PRIMARY KEY, building TEXT, resource TEXT, start TEXT, end TEXT,
        alarm_time TEXT, rule TEXT, peak_z REAL, excess REAL, excess_pct REAL, kind TEXT,
        status TEXT DEFAULT 'OPEN', note TEXT DEFAULT '', decided_at TEXT);
    CREATE TABLE decisions(id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, alert_id TEXT,
        decision TEXT, note TEXT);
    """)
    for a in ACTIONS_TRUTH:
        c.execute("INSERT INTO actions VALUES (?,?,?,?,?,?,?,?,?,?)",
                  (a["action_id"], a["title"], a["building"], a["resource"], a["date"].strftime("%Y-%m-%d"),
                   a["cost_inr"], a["claimed_pct"], "Scenario action (labelled dataset)", "scenario",
                   datetime.now().isoformat(timespec="seconds")))
    for _, e in EVENTS.iterrows():
        c.execute("INSERT INTO alerts(alert_id,building,resource,start,end,alarm_time,rule,peak_z,excess,excess_pct,kind) "
                  "VALUES (?,?,?,?,?,?,?,?,?,?,?)",
                  (e.alert_id, e.building, e.resource, str(e.start), str(e.end), str(e.alarm_time), e.rule,
                   float(e.peak_z), float(e.excess), float(e.excess_pct), e.kind))


def load_actions():
    with db() as c:
        df = pd.read_sql("SELECT * FROM actions ORDER BY date", c)
    df["date"] = pd.to_datetime(df["date"])
    return df


def load_alerts():
    with db() as c:
        df = pd.read_sql("SELECT * FROM alerts", c)
    for col in ("start", "end", "alarm_time"):
        df[col] = pd.to_datetime(df[col])
    return df.sort_values("alarm_time").reset_index(drop=True)


def sync_registry():
    ACTION_REGISTRY[:] = [dict(building=r.building, resource=r.resource, date=r.date)
                          for r in load_actions().itertuples()]


def add_action(title, building, resource, date, cost_inr, claimed_pct, notes=""):
    date = pd.Timestamp(date).normalize()
    with db() as c:
        n = c.execute("SELECT COUNT(*) FROM actions").fetchone()[0]
        aid = f"A{n + 1}"
        while c.execute("SELECT 1 FROM actions WHERE action_id=?", (aid,)).fetchone():
            n += 1; aid = f"A{n + 1}"
        c.execute("INSERT INTO actions VALUES (?,?,?,?,?,?,?,?,?,?)",
                  (aid, title.strip(), building, resource, date.strftime("%Y-%m-%d"), float(cost_inr or 0),
                   float(claimed_pct or 0), notes, "logged", datetime.now().isoformat(timespec="seconds")))
    sync_registry()
    # a new action can only change OTHER actions' comparison groups — drop just those cached results
    acts = load_actions().set_index("action_id")
    for key in list(MV_CACHE):
        r = MV_CACHE[key]
        if "controls" not in r or key[0] not in acts.index:
            continue
        a = acts.loc[key[0]]
        if comparison_buildings(a.resource, a.building, a.date, key[1]) != r["controls"]:
            del MV_CACHE[key]
    return aid


def decide_alert(alert_id, decision, note=""):
    ts = datetime.now().isoformat(timespec="seconds")
    with db() as c:
        c.execute("UPDATE alerts SET status=?, note=?, decided_at=? WHERE alert_id=?", (decision, note, ts, alert_id))
        c.execute("INSERT INTO decisions(ts, alert_id, decision, note) VALUES (?,?,?,?)", (ts, alert_id, decision, note))


MV_CACHE = {}


def get_verification(action_id, post_days=MV_POST_DAYS_DEFAULT, as_of=None):
    """Cached verification of a registered action, using only data up to `as_of`."""
    as_of = pd.Timestamp(as_of or DATA_END).normalize()
    a = load_actions().set_index("action_id").loc[action_id]
    if as_of >= a.date + pd.Timedelta(days=int(post_days) - 1):
        as_of = DATA_END          # result only uses data up to the window end → identical for later weeks
    key = (action_id, int(post_days), as_of)
    if key in MV_CACHE:
        return MV_CACHE[key]
    y = daily_series(a.resource, a.building, True).loc[:as_of + pd.Timedelta(hours=23)]
    r = verify_action(a.resource, a.building, a.date, post_days=int(post_days), y=y)
    MV_CACHE[key] = r
    if "estimate_pct" in r:
        with db() as c:
            c.execute("INSERT OR REPLACE INTO verifications VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                      (action_id, int(post_days), str(as_of.date()), r["verdict"], r["estimate_pct"], r["ci_lo_pct"],
                       r["ci_hi_pct"], r["naive_pct"], r["saved"], r["saved_lo"], r["saved_hi"],
                       RESOURCES[a.resource]["unit"], r["n_placebos"], ",".join(r["controls"]),
                       r["excluded_days"], datetime.now().isoformat(timespec="seconds")))
    return r


sync_registry()
for a in ACTIONS_TRUTH:      # seed the cache with the results computed in cell 8, then the other windows
    MV_CACHE[(a["action_id"], MV_POST_DAYS_DEFAULT, DATA_END)] = MV_RESULTS[a["action_id"]]
_t0 = time.time()
for _w in (28, 56):
    for a in ACTIONS_TRUTH:
        get_verification(a["action_id"], _w, DATA_END)
print(f"Pre-computed 28/56-day windows in {time.time() - _t0:.0f}s")
print(f"Store ready: {len(load_actions())} actions, {len(load_alerts())} alerts → {DB_PATH}")

## 10 · Extension: e-waste reuse planner

In [ ]:
# ============================================================
# 10 · EXTENSION — E-WASTE REUSE PLANNER (synthetic inventory; ESTIMATES, never mixed
#      with measured savings)
# ============================================================
_r = np.random.default_rng(SEED + 9)
DEVICE_TYPES = {  # type: (rated W, design life yrs, embodied kgCO2e, mass kg)  — indicative literature-style values
    "Desktop PC": (120, 6, 300, 9), "Monitor": (30, 8, 150, 5), "Projector": (300, 7, 120, 4),
    "Laser printer": (400, 8, 100, 12), "Split AC 1.5T": (1500, 10, 800, 45), "Laptop": (45, 5, 250, 2),
}
TODAY = pd.Timestamp(datetime.now().date())
_inv = []
for b in BLDS:
    for dtype, (w, life, co2, kg) in DEVICE_TYPES.items():
        for k in range(int(_r.integers(2, 7))):
            age_y = float(np.clip(_r.gamma(2.2, life / 2.6), 0.3, 16))
            cond = _r.choice(["working", "faulty", "non-functional"], p=[0.78, 0.15, 0.07])
            _inv.append(dict(device_id=f"{b[-1]}-{dtype.split()[0][:3].upper()}-{k + 1:02d}", building=b, device_type=dtype,
                             purchase_date=(TODAY - pd.Timedelta(days=int(age_y * 365.25))).date(),
                             age_years=round(age_y, 1), design_life_years=life, condition=cond,
                             months_since_service=int(_r.integers(0, 30)), embodied_kgco2e=co2, mass_kg=kg))
INVENTORY = pd.DataFrame(_inv)

PATHS = {   # recovery path: (share of embodied CO2 avoided — ASSUMPTION, conditions allowed)
    "Refurbish & redeploy on campus": (0.70, {"working", "faulty"}),
    "Donate (school / NGO)": (0.65, {"working"}),
    "Harvest parts for spares": (0.40, {"faulty", "non-functional"}),
    "Certified e-waste recycler": (0.15, {"working", "faulty", "non-functional"}),
}


def plan_ewaste(inv):
    rows = []
    for d in inv.itertuples():
        reason = None
        if d.condition == "non-functional":
            reason = "non-functional"
        elif d.age_years > d.design_life_years:
            reason = f"past design life ({d.age_years:.1f} > {d.design_life_years} yrs)"
        elif d.condition == "faulty" and d.months_since_service > 18:
            reason = f"faulty and unserviced {d.months_since_service} months"
        if not reason:
            continue
        if d.condition == "working" and d.age_years <= d.design_life_years + 2:
            path = "Refurbish & redeploy on campus"
        elif d.condition == "working":
            path = "Donate (school / NGO)"
        elif d.age_years <= d.design_life_years + 3:
            path = "Harvest parts for spares"
        else:
            path = "Certified e-waste recycler"
        share = PATHS[path][0]
        rows.append(dict(device_id=d.device_id, building=d.building, device_type=d.device_type, age_years=d.age_years,
                         condition=d.condition, reason=reason, recovery_path=path, mass_kg=d.mass_kg,
                         est_kgco2e_avoided=round(d.embodied_kgco2e * share, 1)))
    return pd.DataFrame(rows)


EWASTE_PLAN = plan_ewaste(INVENTORY)
print(f"E-waste: {len(EWASTE_PLAN)} of {len(INVENTORY)} devices flagged "
      f"({len(EWASTE_PLAN) / len(INVENTORY):.0%}); estimated {EWASTE_PLAN.est_kgco2e_avoided.sum():,.0f} kgCO2e avoided (ESTIMATE)")

## 11 · View helpers

In [ ]:
# ============================================================
# 11 · VIEW HELPERS (shared by the dashboard and the report)
# ============================================================
def fmt_qty(v, resource):
    if resource == "water":
        return f"{v / 1000:,.1f} kL"
    return f"{v:,.0f} {RESOURCES[resource]['unit']}"


_first_sun = EVAL_START + pd.Timedelta(days=(6 - EVAL_START.dayofweek) % 7 + (7 if (6 - EVAL_START.dayofweek) % 7 < 6 else 0))
WEEK_ENDS = pd.date_range(_first_sun, DATA_END, freq="7D")
if WEEK_ENDS[-1] != DATA_END:
    WEEK_ENDS = WEEK_ENDS.append(pd.DatetimeIndex([DATA_END]))
ASOF_CHOICES = [(f"Week ending {d:%a %d %b %Y}", d.strftime("%Y-%m-%d")) for d in WEEK_ENDS]


# "Expected" on the dashboard = weather/calendar model + recent-normal level (the same
# baseline the alert detector uses), so deviations mean "unusual vs. recent normal".
ADJ_EXPECTED_D = {}
for _res in ("electricity", "water"):
    for _b in BLDS:
        _obs = HOURLY_OBS[_res][_b]
        _adj = _obs - _debias_hourly(_obs - EXPECTED[(_res, _b)])
        _adj = _adj.where(_obs.notna(), EXPECTED[(_res, _b)])
        ADJ_EXPECTED_D[(_res, _b)] = _adj.clip(lower=0).resample("D").sum()
for _b in BLDS:
    _obs = OBS_WASTE_D[_b]["dry"]
    ADJ_EXPECTED_D[("waste", _b)] = (_obs - _debias(_obs - EXPECTED[("waste", _b)], 28, 7)).clip(lower=0)


def actual_expected_daily(resource, b):
    if resource == "waste":
        act = OBS_WASTE_D[b]["dry"]
    else:
        act = HOURLY_OBS[resource][b].resample("D").sum(min_count=20)
    return act, ADJ_EXPECTED_D[(resource, b)]


def week_table(as_of):
    """Per building & resource: last-7-day actual vs expected (expected = model of normal use)."""
    as_of = pd.Timestamp(as_of)
    s = as_of - pd.Timedelta(days=6)
    rows = []
    for b in BLDS:
        row = {"Building": b}
        for res in RESOURCES:
            act, exp = actual_expected_daily(res, b)
            a, e = act.loc[s:as_of].sum(), exp.loc[s:as_of].sum()
            row[res] = a
            row[res + "_dev"] = (a - e) / e * 100 if e > 0 else np.nan
            wk_sd = SIG[(res, b)]["day"] * np.sqrt(7)          # normal week-to-week variation
            row[res + "_z"] = (a - e) / wk_sd if wk_sd > 0 else 0.0
        row["area"] = float(BUILDINGS.set_index("building").loc[b, "area_sqm"])
        rows.append(row)
    return pd.DataFrame(rows)


def visible_alerts(as_of):
    al = load_alerts()
    return al[al.alarm_time <= pd.Timestamp(as_of) + pd.Timedelta(hours=23, minutes=59)]


def ledger(as_of=None, post_days=MV_POST_DAYS_DEFAULT):
    as_of = pd.Timestamp(as_of or DATA_END)
    rows = []
    for a in load_actions().itertuples():
        r = get_verification(a.action_id, post_days, as_of) if a.date <= as_of else \
            {"verdict": "NOT STARTED", "why": "Action date is after the selected week."}
        unit = RESOURCES[a.resource]["unit"]
        has = "estimate_pct" in r
        cons = r["saved_lo"] if has and r["verdict"] == "VERIFIED SAVING" else 0.0
        rows.append({
            "ID": a.action_id, "Action": a.title, "Building": a.building, "Resource": RESOURCES[a.resource]["label"],
            "Date": a.date.strftime("%Y-%m-%d"), "Claimed %": a.claimed_pct,
            "Naive before/after %": round(r["naive_pct"], 1) if has else None,
            "Verdict": r["verdict"],
            "Estimate %": round(r["estimate_pct"], 1) if has else None,
            "95% CI": f"{r['ci_lo_pct']:.1f} … {r['ci_hi_pct']:.1f}" if has else "—",
            "Conservative saving": fmt_qty(cons, a.resource) if cons > 0 else "—",
            "_cons": cons, "_resource": a.resource, "_cost": a.cost_inr,
            "kgCO₂ (≥)": round(cons * EMISSION_FACTOR) if a.resource == "electricity" and cons > 0 else None,
            "₹ (≥, assumed tariff)": round(cons * TARIFF_INR_PER_KWH) if a.resource == "electricity" and cons > 0 else None,
        })
    return pd.DataFrame(rows)


def verified_totals(led):
    out = {}
    for res in RESOURCES:
        out[res] = float(led.loc[(led._resource == res), "_cons"].sum())
    out["co2_kg"] = out["electricity"] * EMISSION_FACTOR
    out["inr"] = out["electricity"] * TARIFF_INR_PER_KWH
    return out


print("Views ready ·", len(ASOF_CHOICES), "weekly snapshots")

## 12 · Exportable report

In [ ]:
# ============================================================
# 12 · EXPORTABLE REPORT — Markdown + printable HTML (save as PDF) + JSON + CSV ledger
# Rule: only VERIFIED savings are counted, and only at their 95% LOWER bound.
# ============================================================
import html as _html


def build_report(as_of=None, post_days=MV_POST_DAYS_DEFAULT):
    as_of = pd.Timestamp(as_of or DATA_END)
    led = ledger(as_of, post_days)
    tot = verified_totals(led)
    wk = week_table(as_of)
    al = visible_alerts(as_of)
    gen = datetime.now().strftime("%d %b %Y %H:%M")
    L = [f"# GreenLedger — Campus Resource Report",
         f"*Reporting week ending {as_of:%d %b %Y} · generated {gen} · measurement window {post_days} days*", "",
         "## 1. Headline (conservative)", "",
         "Only savings whose entire 95% interval is above zero are counted, at the interval's **lower bound**.", "",
         "| Resource | Verified saving (≥) |", "|---|---|",
         f"| Electricity | {tot['electricity']:,.0f} kWh · ≥ {tot['co2_kg'] / 1000:,.2f} tCO₂ · ≥ ₹{tot['inr']:,.0f}* |",
         f"| Water | {tot['water'] / 1000:,.1f} kL |",
         f"| Dry waste to landfill | {tot['waste']:,.0f} kg |", "",
         f"*CO₂ at {EF_SOURCE}. ₹ uses an ASSUMED tariff of ₹{TARIFF_INR_PER_KWH}/kWh.", "",
         "## 2. This week vs expected use", "",
         "| Building | Electricity | Δ vs expected | Water | Δ | Dry waste | Δ |", "|---|---|---|---|---|---|---|"]
    for r in wk.itertuples():
        L.append(f"| {r.Building} | {r.electricity:,.0f} kWh | {r.electricity_dev:+.1f}% | {r.water / 1000:,.1f} kL | "
                 f"{r.water_dev:+.1f}% | {r.waste:,.0f} kg | {r.waste_dev:+.1f}% |")
    L += ["", "## 3. Unusual changes (alerts)", "",
          f"{len(al)} alerts raised up to this week · open {int((al.status == 'OPEN').sum())} · confirmed "
          f"{int((al.status == 'CONFIRMED').sum())} · dismissed {int((al.status == 'DISMISSED').sum())} · "
          f"turned into actions {int((al.status == 'ACTIONED').sum())}", "",
          "| Alert | Raised | Building | Resource | What we saw | Excess | Status |", "|---|---|---|---|---|---|---|"]
    for e in al.sort_values("excess_pct", ascending=False).head(12).itertuples():
        L.append(f"| {e.alert_id} | {e.alarm_time:%d %b %H:%M} | {e.building} | {RESOURCES[e.resource]['label']} | "
                 f"{e.kind} | {fmt_qty(e.excess, e.resource)} (+{e.excess_pct:.0f}%) | {e.status} |")
    L += ["", "## 4. Action register — verified against a counterfactual", "",
          "| ID | Action | Building | Claimed | Naive before/after | Verdict | Estimate (95% CI) | Counted |",
          "|---|---|---|---|---|---|---|---|"]
    def _num(v, f="{:.1f}%"):
        return f.format(v) if v is not None and not pd.isna(v) else "—"
    for _, r in led.iterrows():
        est = f"{_num(r['Estimate %'])} ({r['95% CI']})" if r["95% CI"] != "—" else "—"
        L.append(f"| {r['ID']} | {r['Action']} | {r['Building']} | {r['Claimed %']:.0f}% | "
                 f"{_num(r['Naive before/after %'])} | {r['Verdict']} | {est} | {r['Conservative saving']} |")
    notp = led[led.Verdict == "NOT PROVEN"]
    L += ["", "## 5. What we did NOT claim", ""]
    if len(notp):
        for _, r in notp.iterrows():
            L.append(f"- **{r['Action']}** ({r['Building']}): claimed {r['Claimed %']:.0f}%, naive before/after "
                     f"{_num(r['Naive before/after %'])}, but the 95% interval {r['95% CI']} includes zero → no saving counted.")
    else:
        L.append("- Nothing excluded this period.")
    L += ["", "## 6. How much to trust these numbers", "",
          f"- Savings verifier, tested on {MV_VALID_SUMMARY['cases']} injected effects in real data: 95% interval "
          f"coverage {MV_VALID_SUMMARY['ci_coverage']:.0%}; false savings claims on zero-effect actions "
          f"{MV_VALID_SUMMARY['false_claims_on_zero_effect']} (naive before/after: {MV_VALID_SUMMARY['naive_false_claims_on_zero_effect']}).",
          f"- Alert detector, held-out labelled set: caught {DETECT_SUMMARY['detected']} of {DETECT_SUMMARY['injected']} "
          f"injected problems; {DETECT_SUMMARY['other_alerts_per_series_month']} other alerts per meter per month.",
          "", "## 7. Method & limitations", "",
          "- Counterfactual baseline: LightGBM on 365 days before each action using untreated comparison buildings, "
          "day-of-week and temperature; uncertainty from placebo-in-time (fake action dates) with bias correction.",
          "- Days flagged as abnormal (leaks, faults) are excluded from verification (IPMVP non-routine events).",
          "- Electricity and water are REAL hourly meter data (Building Data Genome 2, 5 classroom buildings); "
          "dry-waste data are SYNTHETIC; interventions and anomalies are injected with known truth for evaluation.",
          "- Small effects (<~5%) need longer measurement windows to be confirmed; NOT PROVEN ≠ 'no effect'.",
          "- CO₂ is reported for electricity only; no CO₂ is claimed for water or waste.", ""]
    md = "\n".join(L)

    def md_table_to_html(lines):
        out, in_tab = [], False
        for ln in lines:
            if ln.startswith("|"):
                cells = [c.strip() for c in ln.strip("|").split("|")]
                if set("".join(cells)) <= set("-: "):
                    continue
                tag = "th" if not in_tab else "td"
                if not in_tab:
                    out.append("<table>"); in_tab = True
                out.append("<tr>" + "".join(f"<{tag}>{_fmt_inline(c)}</{tag}>" for c in cells) + "</tr>")
            else:
                if in_tab:
                    out.append("</table>"); in_tab = False
                if ln.startswith("# "):
                    out.append(f"<h1>{_fmt_inline(ln[2:])}</h1>")
                elif ln.startswith("## "):
                    out.append(f"<h2>{_fmt_inline(ln[3:])}</h2>")
                elif ln.startswith("- "):
                    out.append(f"<li>{_fmt_inline(ln[2:])}</li>")
                elif ln.strip():
                    out.append(f"<p>{_fmt_inline(ln)}</p>")
        if in_tab:
            out.append("</table>")
        return "\n".join(out)

    body = md_table_to_html(L)
    html_doc = f"""<!doctype html><html><head><meta charset="utf-8"><title>GreenLedger report</title>
<style>body{{font-family:Inter,Segoe UI,Arial,sans-serif;max-width:980px;margin:32px auto;color:#14202e;line-height:1.5;padding:0 18px}}
h1{{font-size:26px;margin-bottom:4px}}h2{{font-size:18px;margin-top:28px;border-bottom:2px solid #10B981;padding-bottom:4px}}
table{{border-collapse:collapse;width:100%;font-size:13px;margin:10px 0}}th,td{{border:1px solid #d5dde6;padding:6px 8px;text-align:left}}
th{{background:#eef3f7}}li{{margin:4px 0}}@media print{{body{{margin:0}}}}</style></head><body>{body}</body></html>"""
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    paths = {
        "md": os.path.join(OUT_DIR, f"greenledger_report_{stamp}.md"),
        "html": os.path.join(OUT_DIR, f"greenledger_report_{stamp}.html"),
        "json": os.path.join(OUT_DIR, f"greenledger_report_{stamp}.json"),
        "csv": os.path.join(OUT_DIR, f"greenledger_ledger_{stamp}.csv"),
    }
    open(paths["md"], "w", encoding="utf-8").write(md)
    open(paths["html"], "w", encoding="utf-8").write(html_doc)
    pub = led.drop(columns=[c for c in led.columns if c.startswith("_")])
    pub.to_csv(paths["csv"], index=False)
    json.dump({"as_of": str(as_of.date()), "post_days": post_days, "verified_totals_lower_bound": tot,
               "emission_factor": EF_SOURCE, "tariff_inr_per_kwh_assumed": TARIFF_INR_PER_KWH,
               "ledger": pub.to_dict(orient="records"), "detector_eval": DETECT_SUMMARY,
               "verifier_validation": MV_VALID_SUMMARY}, open(paths["json"], "w"), indent=2, default=str)
    return md, paths


def _fmt_inline(t):
    t = _html.escape(t)
    while "**" in t:
        t = t.replace("**", "<b>", 1).replace("**", "</b>", 1)
    while t.count("*") >= 2:
        t = t.replace("*", "<i>", 1).replace("*", "</i>", 1)
    return t


_md, _paths = build_report()
print("Report test OK:", {k: os.path.basename(v) for k, v in _paths.items()})

## 13 · GreenLedger app (Gradio)

In [ ]:
# ============================================================
# 13 · GREENLEDGER UI (Gradio) — tabs map 1:1 to the problem statement's outputs
# Works with Gradio 4.x / 5.x / 6.x (CSS injected via HTML, no theme/css kwargs).
# ============================================================
import gradio as gr
from plotly.subplots import make_subplots

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@500;700&family=Space+Grotesk:wght@600;700&display=swap');
:root{--bg:#0B1220;--s1:#111A2C;--s2:#182238;--s3:#1F2A44;--bd:#2A3A5C;--tx:#E6EDF7;--tx2:#B6C2D6;--mu:#8794AB;
--em:#10B981;--em2:#34D399;--am:#F59E0B;--am2:#FBBF24;--rd:#EF4444;--rd2:#F87171;--bl:#3B82F6;--bl2:#60A5FA;}
html,body,gradio-app,.gradio-container{background:var(--bg)!important;color:var(--tx)!important;font-family:Inter,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif!important}
.gradio-container{max-width:1500px!important;margin:0 auto!important}
.gradio-container *{--body-background-fill:var(--bg);--background-fill-primary:var(--s1);--background-fill-secondary:var(--s2);
--block-background-fill:var(--s1);--block-border-color:var(--bd);--border-color-primary:var(--bd);--input-background-fill:var(--s2);
--body-text-color:var(--tx);--block-label-text-color:var(--tx2);--block-title-text-color:var(--tx2);--color-accent:var(--em);
--table-even-background-fill:var(--s1);--checkbox-label-background-fill:var(--s2);
--checkbox-label-background-fill-selected:rgba(16,185,129,.18);--checkbox-label-text-color:var(--tx);
--checkbox-label-text-color-selected:#fff;--checkbox-label-border-color:var(--bd);--block-label-background-fill:var(--s2);
--block-label-text-color:var(--tx2);--block-label-border-color:var(--bd);--input-placeholder-color:var(--mu);--table-odd-background-fill:var(--s2);--table-row-focus:var(--s3);--neutral-50:var(--s1);
--neutral-100:var(--s2);--neutral-200:var(--bd);--neutral-700:var(--tx);--neutral-800:var(--tx);--neutral-900:#fff}
.gl-head{display:flex;justify-content:space-between;align-items:center;gap:16px;flex-wrap:wrap;padding:22px 28px;margin:14px 0 10px;
background:linear-gradient(120deg,#111A2C,#16304a);border:1px solid var(--bd);border-radius:14px}
.gl-brand{display:flex;gap:14px;align-items:center}.gl-logo{width:50px;height:50px;border-radius:12px;background:linear-gradient(135deg,#10B981,#047857);
display:flex;align-items:center;justify-content:center;font:700 26px 'Space Grotesk',Inter,system-ui,-apple-system,'Segoe UI',sans-serif;color:#fff}
.gl-title{font:700 26px 'Space Grotesk',Inter,system-ui,-apple-system,'Segoe UI',sans-serif;color:#fff;letter-spacing:-.02em}.gl-sub{font:500 12px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--tx2);margin-top:4px;letter-spacing:.04em}
.gl-pill{font:700 11px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;padding:8px 12px;border-radius:8px;border:1px solid rgba(16,185,129,.45);background:rgba(16,185,129,.12);color:var(--em2)}
.gl-intro{background:var(--s1);border:1px solid var(--bd);border-left:4px solid var(--em);border-radius:10px;padding:14px 18px;margin:12px 0;font-size:14px;color:var(--tx2);line-height:1.6}
.gl-intro b{color:#fff}
.gl-cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:12px;margin:12px 0}
.gl-card{background:linear-gradient(160deg,var(--s1),var(--s2));border:1px solid var(--bd);border-left:4px solid var(--em);border-radius:12px;padding:14px 16px}
.gl-card.warn{border-left-color:var(--am)}.gl-card.bad{border-left-color:var(--rd)}.gl-card.info{border-left-color:var(--bl)}.gl-card.mute{border-left-color:#5E6B85}
.gl-v{font:700 26px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:#fff;letter-spacing:-.02em}.gl-u{font-size:13px;color:var(--tx2);margin-left:4px}
.gl-l{font:700 11px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--tx2);text-transform:uppercase;letter-spacing:.08em;margin-top:8px}
.gl-h{font-size:12px;color:var(--mu);margin-top:4px;line-height:1.45}
.gl-verdict{border-radius:14px;padding:20px 24px;margin:10px 0;border:1px solid var(--bd)}
.gl-verdict .big{font:700 30px 'Space Grotesk',Inter,system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:-.01em}.gl-verdict .num{font:700 20px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-top:6px;color:#fff}
.gl-verdict p{color:var(--tx2);font-size:14px;line-height:1.6;margin:10px 0 0}
.gl-ok{background:rgba(16,185,129,.12);border-color:rgba(16,185,129,.5)}.gl-ok .big{color:var(--em2)}
.gl-no{background:rgba(245,158,11,.10);border-color:rgba(245,158,11,.45)}.gl-no .big{color:var(--am2)}
.gl-bad{background:rgba(239,68,68,.10);border-color:rgba(239,68,68,.45)}.gl-bad .big{color:var(--rd2)}
.gl-wait{background:var(--s2)}.gl-wait .big{color:var(--tx2)}
table.gl-t{width:100%;border-collapse:collapse;font:13px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin:8px 0}
table.gl-t th{background:var(--s3);color:#fff;text-align:left;padding:9px 10px;font-size:11px;text-transform:uppercase;letter-spacing:.06em}
table.gl-t td{padding:8px 10px;border-bottom:1px solid var(--bd);color:var(--tx)}
.z-hi{background:rgba(239,68,68,.22)!important;color:#FECACA!important}.z-lo{background:rgba(16,185,129,.20)!important;color:#A7F3D0!important}
.gl-sec{font:700 12px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--tx2);letter-spacing:.12em;text-transform:uppercase;margin:22px 0 8px}
.gl-note{font-size:12.5px;color:var(--mu);font-style:italic;margin:6px 0}
.gl-msg{font:600 12.5px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--em2);padding:8px 12px;background:rgba(16,185,129,.1);border:1px solid rgba(16,185,129,.35);border-radius:8px;display:inline-block}
.gl-msg.warn{color:var(--am2);background:rgba(245,158,11,.1);border-color:rgba(245,158,11,.4)}
.tabs button{font:600 12.5px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace!important;letter-spacing:.06em!important;text-transform:uppercase!important}
.tabs button.selected{color:var(--em2)!important;border-color:var(--em2)!important}
button.primary{background:linear-gradient(135deg,#10B981,#059669)!important;color:#fff!important;border:none!important;font-weight:700!important}
button.secondary{background:var(--s2)!important;color:var(--tx)!important;border:1px solid var(--bd)!important;font-weight:600!important}
button.secondary:hover{border-color:var(--em2)!important;color:var(--em2)!important}
.gl-foot{margin:36px 0 18px;padding-top:14px;border-top:1px solid var(--bd);display:flex;justify-content:space-between;flex-wrap:wrap;gap:10px;
font:500 11px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--mu);letter-spacing:.06em;text-transform:uppercase}
</style>
"""

PL = dict(plot_bgcolor="#111A2C", paper_bgcolor="#111A2C",
          font=dict(family="Inter, sans-serif", size=12, color="#E6EDF7"),
          margin=dict(t=56, b=44, l=60, r=24),
          legend=dict(orientation="h", y=-0.18, bgcolor="rgba(0,0,0,0)"),
          hoverlabel=dict(bgcolor="#1F2A44", font=dict(color="#fff")))
GRID = dict(gridcolor="#243352", zerolinecolor="#324369", linecolor="#324369")


def _fig(fig, title, h=380):
    fig.update_layout(title=dict(text=title, font=dict(size=15, color="#fff")), height=h, **PL)
    fig.update_xaxes(**GRID); fig.update_yaxes(**GRID)
    return fig


def card(v, label, help_="", cls="", unit=""):
    u = f'<span class="gl-u">{unit}</span>' if unit else ""
    h = f'<div class="gl-h">{help_}</div>' if help_ else ""
    return f'<div class="gl-card {cls}"><div class="gl-v">{v}{u}</div><div class="gl-l">{label}</div>{h}</div>'


def intro(text):
    return f'<div class="gl-intro">{text}</div>'


def msg(text, warn=False):
    return f'<div class="gl-msg{" warn" if warn else ""}">{text}</div>'


def _asof(v):
    return pd.Timestamp(v or ASOF_CHOICES[-1][1])


# ---------------- OVERVIEW ----------------
def ui_overview(as_of_v, window):
    as_of = _asof(as_of_v)
    wk = week_table(as_of)
    led = ledger(as_of, int(window))
    tot = verified_totals(led)
    al = visible_alerts(as_of)
    s7 = as_of - pd.Timedelta(days=6)
    new_al = int(((al.alarm_time >= s7) & (al.alarm_time <= as_of + pd.Timedelta(days=1))).sum())
    cards = []
    for res in RESOURCES:
        a = wk[res].sum()
        e = sum(actual_expected_daily(res, b)[1].loc[s7:as_of].sum() for b in BLDS)
        dev = (a - e) / e * 100 if e > 0 else 0
        cls = "bad" if dev > 10 else ("" if dev < -3 else "info")
        cards.append(card(fmt_qty(a, res), f"{RESOURCES[res]['label']} · this week",
                          f"{dev:+.1f}% vs recent normal (weather-adjusted)", cls))
    cards.append(card(f"{wk['electricity'].sum() * EMISSION_FACTOR / 1000:,.1f}", "Grid CO₂ · this week",
                      "from electricity · CEA factor 0.710", "mute", "t"))
    cards.append(card(f"{int((al.status == 'OPEN').sum())}", "Open alerts", f"{new_al} raised this week", "warn"))
    cards.append(card(f"{tot['electricity']:,.0f}", "Verified saving · electricity",
                      f"95% lower bound only · ≥ {tot['co2_kg'] / 1000:,.1f} tCO₂", "", "kWh"))
    cards.append(card(f"{tot['water'] / 1000:,.1f}", "Verified saving · water", "95% lower bound only", "", "kL"))
    cards.append(card(f"{tot['waste']:,.0f}", "Verified saving · dry waste", "95% lower bound only", "", "kg"))
    cards_html = '<div class="gl-cards">' + "".join(cards) + "</div>"

    # campus chart: last 12 weeks, actual vs expected, alerts marked
    s = as_of - pd.Timedelta(days=83)
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=[f"{RESOURCES[r]['label']} ({'kL' if r == 'water' else RESOURCES[r]['unit']}/day, campus)" for r in RESOURCES])
    for i, res in enumerate(RESOURCES, 1):
        A = sum(actual_expected_daily(res, b)[0].loc[s:as_of].fillna(0) for b in BLDS)
        E = sum(actual_expected_daily(res, b)[1].loc[s:as_of].fillna(0) for b in BLDS)
        k = 1000 if res == "water" else 1
        fig.add_trace(go.Scatter(x=E.index, y=E / k, name="Expected (recent normal)", line=dict(color="#8794AB", dash="dot", width=2),
                                 showlegend=(i == 1), hovertemplate="%{x|%d %b}: %{y:,.1f}<extra>expected</extra>"), row=i, col=1)
        fig.add_trace(go.Scatter(x=A.index, y=A / k, name="Actual", line=dict(color=RESOURCES[res]["color"], width=2.5),
                                 showlegend=(i == 1), hovertemplate="%{x|%d %b}: %{y:,.1f}<extra>actual</extra>"), row=i, col=1)
        ar = al[(al.resource == res) & (al.alarm_time >= s)]
        if len(ar):
            d = ar.alarm_time.dt.normalize()
            fig.add_trace(go.Scatter(x=d, y=[A.reindex([x]).fillna(0).iloc[0] / k for x in d], mode="markers",
                                     marker=dict(symbol="triangle-up", size=12, color="#F87171", line=dict(color="#fff", width=1)),
                                     name="Alert raised", showlegend=(i == 1), text=ar.alert_id + " · " + ar.building,
                                     hovertemplate="%{text}<extra></extra>"), row=i, col=1)
        for a in load_actions().itertuples():
            if a.resource == res and s <= a.date <= as_of:
                fig.add_vline(x=a.date, line=dict(color="#34D399", width=1, dash="dash"), row=i, col=1)
    _fig(fig, "Campus totals — last 12 weeks (green dashed = action logged · red ▲ = alert)", 640)
    for ann in fig.layout.annotations:
        ann.font = dict(size=12, color="#B6C2D6")

    # scorecard
    rows = []
    for r in wk.itertuples():
        def cell(v, dev, z, res):
            cls = "z-hi" if z > 2 else ("z-lo" if z < -2 else "")
            return f'<td class="{cls}">{fmt_qty(v, res)} <span style="opacity:.75">({dev:+.0f}%)</span></td>'
        rows.append(f"<tr><td><b>{r.Building}</b></td>"
                    + cell(r.electricity, r.electricity_dev, r.electricity_z, "electricity")
                    + cell(r.water, r.water_dev, r.water_z, "water")
                    + cell(r.waste, r.waste_dev, r.waste_z, "waste")
                    + f"<td>{r.electricity / r.area:.2f}</td>"
                    + f"<td>{int(((al.building == r.Building) & (al.status == 'OPEN')).sum())}</td></tr>")
    score = ('<table class="gl-t"><tr><th>Building</th><th>Electricity (Δ vs normal)</th><th>Water</th><th>Dry waste</th>'
             '<th>kWh / m² · week</th><th>Open alerts</th></tr>' + "".join(rows) + "</table>"
             '<div class="gl-note">Red = significantly ABOVE normal (> 2σ of usual week-to-week variation) · '
             'Green = significantly below · uncoloured = within normal variation, even if the % looks large.</div>')
    return cards_html, fig, score


# ---------------- ALERTS ----------------
NEXT_STEP = {
    "leak": "Walk the building between 01:00 and 04:00: check urinal flush valves, cistern floats, tank overflow and irrigation lines.",
    "valve": "Check for a stuck valve, overflowing tank or open hose; compare with the cleaning / event schedule.",
    "night": "Audit BMS schedules and switch-off routines; check labs and server rooms left running.",
    "weekend": "Check whether a weekend HVAC or lighting override was left on after an event.",
    "drift": "Inspect HVAC plant (chiller/AHU) for degraded operation; review set-points that may have been changed.",
    "waste": "Check event bookings and bin segregation for that day; inspect bins for mixed recyclables.",
    "spike": "Look for one-off equipment use; confirm with the building manager.",
}


def _next_step(kind):
    k = kind.lower()
    for key, word in (("leak", "leak"), ("valve", "valve"), ("weekend", "weekend"), ("night", "night"),
                      ("sustained", "drift"), ("waste", "waste")):
        if word in k:
            return NEXT_STEP[key]
    return NEXT_STEP["spike"]


def _alerts_view(as_of_v, res_f, status_f):
    al = visible_alerts(_asof(as_of_v))
    if res_f and res_f != "All":
        al = al[al.resource == res_f]
    if status_f and status_f != "All":
        al = al[al.status == status_f]
    al = al.sort_values("alarm_time", ascending=False)
    tbl = pd.DataFrame({
        "Alert": al.alert_id, "Raised": al.alarm_time.dt.strftime("%d %b %H:%M"), "Building": al.building,
        "Resource": al.resource.map(lambda r: RESOURCES[r]["label"]), "What we saw": al.kind,
        "Excess": [fmt_qty(e, r) for e, r in zip(al.excess, al.resource)],
        "Excess %": al.excess_pct.round(0), "Peak σ": al.peak_z, "Status": al.status})
    choices = [(f"{r.alert_id} · {r.building} · {RESOURCES[r.resource]['label']} · {r.alarm_time:%d %b}", r.alert_id)
               for r in al.itertuples()]
    return tbl, choices


def ui_alerts(as_of_v, res_f="All", status_f="All"):
    tbl, choices = _alerts_view(as_of_v, res_f, status_f)
    first = choices[0][1] if choices else None
    fig, expl = ui_alert_detail(first)
    return tbl, gr.update(choices=choices, value=first), fig, expl


def ui_alert_detail(alert_id):
    if not alert_id:
        return _fig(go.Figure(), "Select an alert"), intro("No alerts for this selection.")
    e = load_alerts().set_index("alert_id").loc[alert_id]
    res, b = e.resource, e.building
    fig = go.Figure()
    if res == "waste":
        s, t = e.start - pd.Timedelta(days=21), e.end + pd.Timedelta(days=14)
        act = OBS_WASTE_D[b]["dry"].loc[s:t]
        exp = ADJ_EXPECTED_D[("waste", b)].loc[s:t]
        sd = SIG[("waste", b)]["day"]
        lo_, hi_ = (exp - 2 * sd).clip(lower=0), exp + 2 * sd
        unit = "kg/day"
    else:
        s, t = e.start - pd.Timedelta(days=4), e.end + pd.Timedelta(days=2)
        obs = HOURLY_OBS[res][b]
        expH = (obs - _debias_hourly(obs - EXPECTED[(res, b)])).clip(lower=0)
        act, exp = obs.loc[s:t], expH.loc[s:t]
        sdh = SIG[(res, b)]["hour"].reindex((exp.index.dayofweek >= 5).astype(int) * 24 + exp.index.hour).values
        lo_, hi_ = (exp - 2 * sdh).clip(lower=0), exp + 2 * sdh
        unit = "kWh/h" if res == "electricity" else "L/h"
    fig.add_trace(go.Scatter(x=hi_.index, y=hi_, line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=lo_.index, y=lo_, fill="tonexty", fillcolor="rgba(135,148,171,0.18)", line=dict(width=0),
                             name="Normal range (±2σ)", hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=exp.index, y=exp, name="Expected", line=dict(color="#8794AB", dash="dot")))
    fig.add_trace(go.Scatter(x=act.index, y=act, name="Actual", line=dict(color=RESOURCES[res]["color"], width=2.4)))
    fig.add_vrect(x0=e.start, x1=e.end, fillcolor="rgba(239,68,68,0.15)", line_width=0)
    fig.add_vline(x=e.alarm_time, line=dict(color="#F87171", dash="dash"))
    _fig(fig, f"{alert_id} · {b} · {RESOURCES[res]['label']} ({unit}) — red band = flagged period, dashed = alert raised", 400)
    dur_h = (e.end - e.start) / pd.Timedelta(hours=1)
    dur = f"{dur_h:.0f} hours" if dur_h < 48 else f"{dur_h / 24:.1f} days"
    co2 = f" ≈ {e.excess * EMISSION_FACTOR:,.0f} kgCO₂ and ₹{e.excess * TARIFF_INR_PER_KWH:,.0f} (assumed tariff)." if res == "electricity" else "."
    rule = {"spike": "hourly spike test", "cusum": "drift test (CUSUM on daily excess)", "mnf": "minimum-night-flow leak test",
            "h6": "6-hour sustained-flow test", "day": "daily volume test"}.get(e.rule, e.rule)
    html_ = intro(
        f"<b>{e.kind}.</b> {b} used <b>{fmt_qty(e.excess, res)} more than expected</b> (+{e.excess_pct:.0f}%) over {dur}, "
        f"from {e.start:%a %d %b %H:%M} to {e.end:%a %d %b %H:%M}{co2} Peak deviation {e.peak_z:.1f}σ, raised by the "
        f"{rule} at {e.alarm_time:%a %d %b %H:%M}.<br><b>Suggested check:</b> {_next_step(e.kind)}<br>"
        f"<b>Status:</b> {e.status}{(' — ' + e.note) if e.note else ''}")
    return fig, html_


def ui_decide(alert_id, decision, note, as_of_v, res_f, status_f):
    if not alert_id:
        return msg("Pick an alert first.", True), *ui_alerts(as_of_v, res_f, status_f), _decision_log()
    decide_alert(alert_id, decision, note or "")
    tbl, dd, fig, expl = ui_alerts(as_of_v, res_f, status_f)
    fig, expl = ui_alert_detail(alert_id)
    return msg(f"{alert_id} → {decision}"), tbl, gr.update(value=alert_id), fig, expl, _decision_log()


def _decision_log():
    with db() as c:
        return pd.read_sql("SELECT ts AS 'When', alert_id AS 'Alert', decision AS 'Decision', note AS 'Note' "
                           "FROM decisions ORDER BY id DESC LIMIT 30", c)


def ui_alert_to_action(alert_id):
    if not alert_id:
        return (msg("Pick an alert first.", True),) + (gr.update(),) * 5
    e = load_alerts().set_index("alert_id").loc[alert_id]
    decide_alert(alert_id, "ACTIONED", "converted to an action in the register")
    title = f"Fix: {e.kind.split(' — ')[0]} ({alert_id})"
    return (msg(f"{alert_id} → ACTIONED. Form pre-filled in Action Register."), gr.update(value=title),
            gr.update(value=e.building), gr.update(value=e.resource),
            gr.update(value=(e.end.normalize() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")), gr.update(selected="register"))


# ---------------- ACTION REGISTER ----------------
def _ledger_view(as_of_v, window):
    led = ledger(_asof(as_of_v), int(window))
    return led.drop(columns=[c for c in led.columns if c.startswith("_")])


def _action_choices():
    return [(f"{a.action_id} · {a.title} · {a.building}", a.action_id) for a in load_actions().itertuples()]


def ui_log_action(title, building, resource, date, cost, claimed, notes, as_of_v, window):
    try:
        if not title or not str(title).strip():
            raise ValueError("Give the action a title.")
        d = pd.Timestamp(str(date).strip())
        if d < IDX.min() or d > DATA_END:
            raise ValueError(f"Date must be between {IDX.min():%Y-%m-%d} and {DATA_END:%Y-%m-%d} (the meter data range).")
        aid = add_action(title, building, resource, d, cost, claimed, notes)
        m = msg(f"Logged {aid}. Open 'Verify savings' to test it — it needs {window} days of data after {d:%d %b %Y}.")
    except Exception as ex:
        m = msg(f"Not logged: {ex}", True)
    return m, _ledger_view(as_of_v, window), gr.update(choices=_action_choices())


# ---------------- VERIFY ----------------
VERDICT_CLASS = {"VERIFIED SAVING": "gl-ok", "NOT PROVEN": "gl-no", "VERIFIED INCREASE": "gl-bad"}


def ui_verify(action_id, as_of_v, window):
    if not action_id:
        return intro("Pick an action."), _fig(go.Figure(), ""), _fig(go.Figure(), ""), ""
    a = load_actions().set_index("action_id").loc[action_id]
    as_of = _asof(as_of_v)
    if a.date > as_of:
        r = {"verdict": "NOT STARTED", "why": f"This action is dated {a.date:%d %b %Y}, after the selected week."}
    else:
        r = get_verification(action_id, int(window), as_of)
    label, unit = RESOURCES[a.resource]["label"], RESOURCES[a.resource]["unit"]
    if "estimate_pct" not in r:
        card_ = f'<div class="gl-verdict gl-wait"><div class="big">{r["verdict"]}</div><p>{r["why"]}</p></div>'
        return card_, _fig(go.Figure(), "Waiting for data"), _fig(go.Figure(), ""), ""
    est, lo, hi = r["estimate_pct"], r["ci_lo_pct"], r["ci_hi_pct"]
    big = {"VERIFIED SAVING": "✓ VERIFIED SAVING", "NOT PROVEN": "○ NOT PROVEN — no saving claimed",
           "VERIFIED INCREASE": "✕ USE WENT UP"}[r["verdict"]]
    extra = ""
    if r["verdict"] == "VERIFIED SAVING":
        extra = f" · counted: ≥ {fmt_qty(r['saved_lo'], a.resource)}"
        if a.resource == "electricity":
            extra += f" · ≥ {r['saved_lo'] * EMISSION_FACTOR / 1000:,.2f} tCO₂ · ≥ ₹{r['saved_lo'] * TARIFF_INR_PER_KWH:,.0f}*"
    card_ = (f'<div class="gl-verdict {VERDICT_CLASS[r["verdict"]]}"><div class="big">{big}</div>'
             f'<div class="num">{est:+.1f}%  (95% CI {lo:.1f}% … {hi:.1f}%){extra}</div>'
             f'<p>{explain_verdict(r, unit if a.resource != "water" else "L", label)}</p></div>')
    comp = ('<div class="gl-cards">'
            + card(f"{a.claimed_pct:.0f}%", "Claimed by the team", "what the proposal / vendor promised", "mute")
            + card(f"{r['naive_pct']:.1f}%", "Naive before/after", "4 weeks before vs. the window after — ignores season & calendar", "warn")
            + card(f"{est:.1f}%", "GreenLedger estimate", f"95% CI {lo:.1f} … {hi:.1f}% · {r['n_placebos']} placebo tests", "")
            + card(f"{len(r['controls'])}", "Comparison buildings", ", ".join(r["controls"]) or "calendar baseline", "info")
            + card(f"{r['excluded_days']}", "Abnormal days excluded", "alert days removed (IPMVP non-routine)", "mute")
            + "</div>")
    if a.resource == "electricity" and r["verdict"] == "VERIFIED SAVING":
        comp += '<div class="gl-note">* ₹ uses an ASSUMED tariff of ₹%.1f/kWh — set your campus tariff in cell 1.</div>' % TARIFF_INR_PER_KWH

    # chart 1: actual vs counterfactual with CI band
    T = r["action_date"]
    y = daily_series(a.resource, a.building, True)
    k = 1000 if a.resource == "water" else 1
    pre = y.loc[T - pd.Timedelta(days=56): T - pd.Timedelta(days=1)] / k
    pred, act = r["pred"] / k, r["act"] / k
    band_hi = pred * (1 - est / 100) / (1 - hi / 100)
    band_lo = pred * (1 - est / 100) / (1 - lo / 100) if lo < 99 else pred
    up, dn = np.maximum(band_hi, band_lo), np.minimum(band_hi, band_lo)
    f1 = go.Figure()
    f1.add_trace(go.Scatter(x=up.index, y=up, line=dict(width=0), showlegend=False, hoverinfo="skip"))
    f1.add_trace(go.Scatter(x=dn.index, y=dn, fill="tonexty", fillcolor="rgba(135,148,171,0.22)", line=dict(width=0),
                            name="Counterfactual range (95%)", hoverinfo="skip"))
    f1.add_trace(go.Scatter(x=pred.index, y=pred, name="Without the action (counterfactual)", line=dict(color="#CBD5E1", dash="dash", width=2)))
    f1.add_trace(go.Scatter(x=pre.index, y=pre, name="Actual (before)", line=dict(color="#8794AB", width=2)))
    f1.add_trace(go.Scatter(x=act.index, y=act, name="Actual (after)", line=dict(color=RESOURCES[a.resource]["color"], width=2.6)))
    f1.add_vline(x=T, line=dict(color="#34D399", dash="dot"))
    _fig(f1, f"{a.title} · {a.building} — daily {label} ({'kL' if a.resource == 'water' else unit})", 400)

    # chart 2: placebo distribution
    null = np.array(r["null"])
    f2 = go.Figure()
    f2.add_trace(go.Histogram(x=null, nbinsx=18, marker_color="#3B82F6", opacity=0.75, name="Placebo 'effects' (nothing happened)"))
    f2.add_vrect(x0=lo, x1=hi, fillcolor="rgba(16,185,129,0.12)", line_width=0)
    f2.add_vline(x=est, line=dict(color="#34D399", width=3), annotation_text=f"this action: {est:.1f}%",
                 annotation_font_color="#34D399")
    f2.add_vline(x=0, line=dict(color="#8794AB", dash="dot"))
    if not np.isnan(r["naive_pct"]):
        f2.add_vline(x=r["naive_pct"], line=dict(color="#FBBF24", dash="dash"), annotation_text=f"naive: {r['naive_pct']:.1f}%",
                     annotation_position="bottom right", annotation_font_color="#FBBF24")
    _fig(f2, "How big are 'savings' on dates when nothing happened? (placebo test)", 400)
    f2.update_xaxes(title="apparent saving after bias correction, %"); f2.update_yaxes(title="placebo dates")
    return card_, f1, f2, comp


# ---------------- EVIDENCE ----------------
def ui_evidence():
    S, V = DETECT_SUMMARY, MV_VALID_SUMMARY
    cov_cls = "" if V["ci_coverage"] >= 0.9 else "warn"
    cards_ = '<div class="gl-cards">' + "".join([
        card(f"{V['ci_coverage']:.0%}", "Savings CI coverage", f"target 95% · {V['cases']} held-out injected effects in real data "
             f"(interval width calibrated on a separate set)", cov_cls),
        card(V["false_claims_on_zero_effect"], "False savings claims", "on actions with ZERO true effect", ""),
        card(V["naive_false_claims_on_zero_effect"].split(" (")[0], "Naive false claims", "same zero-effect actions, before/after >5%", "bad"),
        card(f"{S['detected']}/{S['injected']}", "Problems caught", f"held-out labelled set · recall {S['recall']:.0%}", ""),
        card(f"{S['other_alerts_per_series_month']}", "Other alerts / meter / month", "includes real unlabelled events in BDG2", "info"),
        card(f"{S['median_delay_h']:.0f} h" if S["median_delay_h"] is not None else "—", "Median time to alert", "from problem start", "mute"),
    ]) + "</div>"
    # validation scatter
    d = MV_VALID_DF
    f = go.Figure()
    for res in RESOURCES:
        x = d[d.resource == res]
        if not len(x):
            continue
        jitter = np.random.default_rng(1).uniform(-0.8, 0.8, len(x))
        f.add_trace(go.Scatter(x=x.true_pct + jitter, y=x.estimate_pct, mode="markers", name=RESOURCES[res]["label"],
                               marker=dict(color=RESOURCES[res]["color"], size=9, line=dict(color="#fff", width=1)),
                               error_y=dict(type="data", symmetric=False, array=x.ci_hi - x.estimate_pct,
                                            arrayminus=x.estimate_pct - x.ci_lo, color="rgba(230,237,247,0.35)", thickness=1.2)))
    f.add_trace(go.Scatter(x=[-3, 28], y=[-3, 28], mode="lines", line=dict(color="#8794AB", dash="dot"), name="perfect"))
    _fig(f, "Verifier validation — true injected effect vs. estimate with 95% CI", 420)
    f.update_xaxes(title="true effect, %"); f.update_yaxes(title="estimated effect, %")
    # scenario table
    sc = []
    for a in ACTIONS_TRUTH:
        r = MV_RESULTS[a["action_id"]]
        sc.append({"ID": a["action_id"], "Action": a["title"], "Claimed %": a["claimed_pct"],
                   "TRUE effect %": round(true_effect_pct(a), 1), "Naive %": round(r["naive_pct"], 1),
                   "GreenLedger %": round(r["estimate_pct"], 1), "95% CI": f"{r['ci_lo_pct']:.1f} … {r['ci_hi_pct']:.1f}",
                   "Verdict": r["verdict"],
                   "Truth inside CI?": "yes" if r["ci_lo_pct"] <= true_effect_pct(a) <= r["ci_hi_pct"] else "NO"})
    per_kind = (DETECT_PER.groupby("kind").agg(injected=("detected", "size"), caught=("detected", "sum"),
                                               median_delay_h=("delay_h", "median")).reset_index())
    per_kind["recall"] = (per_kind.caught / per_kind.injected).round(2)
    fit = MODEL_FIT_DF.rename(columns={"resource": "Resource", "building": "Building",
                                       "cv_rmse_hourly_pct": "CV(RMSE) hourly %", "nmbe_pct": "Bias NMBE %",
                                       "cv_rmse_daily_pct": "CV(RMSE) daily %"})
    truth = DETECT_PER.copy()
    truth["start"] = truth.start.dt.strftime("%d %b %H:%M")
    return cards_, f, pd.DataFrame(sc), per_kind, truth, fit


PROVENANCE = pd.DataFrame([
    {"Data": "Electricity (hourly kWh)", "Source": f"REAL — Building Data Genome 2, site '{SITE}', 5 college-classroom buildings", "Period": "2016-01-01 → 2017-12-31"},
    {"Data": "Water (hourly litres)", "Source": "REAL — BDG2 water meters, same buildings", "Period": "2016-01-01 → 2017-12-31"},
    {"Data": "Weather (air temperature)", "Source": "REAL — BDG2 weather.csv for the same site", "Period": "2016-01-01 → 2017-12-31"},
    {"Data": "Dry / wet / recyclable waste (kg/day)", "Source": "SYNTHETIC — driven by each building's occupancy pattern", "Period": "same"},
    {"Data": "5 interventions (A1–A5)", "Source": "INJECTED with known true effect (A3 = zero effect)", "Period": "2017"},
    {"Data": f"{len(ANOMALY_TRUTH_DF)} anomalies", "Source": "INJECTED with known start/end — held out from tuning", "Period": "Jul–Dec 2017"},
    {"Data": "E-waste inventory", "Source": "SYNTHETIC (extension module)", "Period": "current"},
])


# ---------------- REPORT / E-WASTE ----------------
def ui_report(as_of_v, window):
    md, paths = build_report(_asof(as_of_v), int(window))
    return md, [paths["html"], paths["md"], paths["csv"], paths["json"]]


def ui_ewaste():
    p = EWASTE_PLAN
    cards_ = '<div class="gl-cards">' + "".join([
        card(f"{len(p)}/{len(INVENTORY)}", "Devices flagged", "non-functional, past design life, or faulty & unserviced", "warn"),
        card(f"{p.mass_kg.sum():,.0f}", "Material in scope", "kg of equipment", "info", "kg"),
        card(f"{p.est_kgco2e_avoided.sum() / 1000:,.1f}", "Embodied CO₂e avoided", "ESTIMATE from assumed recovery shares — never added to measured savings", "mute", "t"),
    ]) + "</div>"
    g = p.groupby("recovery_path").agg(devices=("device_id", "size"), co2=("est_kgco2e_avoided", "sum")).reset_index()
    f = go.Figure(go.Bar(x=g.recovery_path, y=g.co2, marker_color="#3B82F6", text=g.devices.astype(str) + " devices",
                         textposition="outside"))
    _fig(f, "Estimated embodied CO₂e avoided by recovery path (kg, assumption-based)", 380)
    return cards_, f, p


# ---------------- LAYOUT ----------------
HEADER = f"""
<div class="gl-head"><div class="gl-brand"><div class="gl-logo">G</div><div>
<div class="gl-title">GreenLedger</div>
<div class="gl-sub">CAMPUS RESOURCE OPTIMISATION · ELECTRICITY · WATER · WASTE · IC26-05-01</div></div></div>
<div class="gl-pill">REAL METER DATA · {len(BLDS)} BUILDINGS · SAVINGS VERIFIED, NEVER ASSUMED</div></div>"""

_init_asof = ASOF_CHOICES[-3][1]          # a normal pre-Christmas week as the opening view
_init_window = MV_POST_DAYS_DEFAULT
_ov = ui_overview(_init_asof, _init_window)
_al_tbl0, _al_choices0 = _alerts_view(_init_asof, "All", "All")
_al_first0 = _al_choices0[0][1] if _al_choices0 else None
_al_fig0, _al_expl0 = ui_alert_detail(_al_first0)
_init_actions = _action_choices()
_vf = ui_verify("A1", _init_asof, _init_window)
_ev = ui_evidence()
_ew = ui_ewaste()

with gr.Blocks(title="GreenLedger") as demo:
    gr.HTML(CSS + HEADER)
    with gr.Row():
        asof_dd = gr.Dropdown(ASOF_CHOICES, value=_init_asof, label="Viewing campus as of", scale=3,
                              info="Step through time: alerts and verdicts only use data available that week")
        window_rb = gr.Radio([28, 42, 56], value=_init_window, label="Savings measurement window (days)", scale=2)

    with gr.Tabs() as tabs:
        with gr.Tab("Overview", id="overview"):
            gr.HTML(intro("<b>One view of electricity, water and waste.</b> Each reading is compared with what the "
                          "building normally uses for this weather and day (recent normal). Savings are counted only "
                          "after verification, and only at the 95% lower bound."))
            ov_cards = gr.HTML(_ov[0])
            ov_plot = gr.Plot(_ov[1])
            gr.HTML('<div class="gl-sec">Building scorecard · this week</div>')
            ov_score = gr.HTML(_ov[2])

        with gr.Tab("Alerts", id="alerts"):
            gr.HTML(intro("<b>Unusual changes, explained.</b> Electricity: hourly spike + drift tests. Water: "
                          "<b>minimum night flow</b> (the standard leak indicator) + sustained-flow test. Waste: daily volume. "
                          "Only excess use is alerted. Staff confirm, dismiss or turn an alert into an action."))
            with gr.Row():
                res_f = gr.Dropdown(["All", "electricity", "water", "waste"], value="All", label="Resource", scale=1)
                st_f = gr.Dropdown(["All", "OPEN", "CONFIRMED", "DISMISSED", "ACTIONED"], value="All", label="Status", scale=1)
            al_tbl = gr.Dataframe(_al_tbl0, label="Alerts raised up to the selected week", interactive=False, wrap=True)
            al_dd = gr.Dropdown(choices=_al_choices0, value=_al_first0, label="Inspect alert")
            al_plot = gr.Plot(_al_fig0)
            al_expl = gr.HTML(_al_expl0)
            with gr.Row():
                note_tb = gr.Textbox(label="Note (optional)", placeholder="e.g. facilities found a running urinal valve", scale=3)
                conf_btn = gr.Button("✓ Confirm problem", variant="primary", scale=1)
                dism_btn = gr.Button("✕ Dismiss (false alarm)", scale=1)
                act_btn = gr.Button("→ Turn into action", scale=1)
            al_msg = gr.HTML()
            gr.HTML('<div class="gl-sec">Decision log</div>')
            dec_tbl = gr.Dataframe(_decision_log(), interactive=False)

        with gr.Tab("Action Register", id="register"):
            gr.HTML(intro("<b>Every intervention, logged and tested.</b> The ledger shows what was claimed, what a naive "
                          "before/after comparison says, and what survives a counterfactual test. Actions still inside their "
                          "measurement window show AWAITING DATA — nothing is claimed early."))
            reg_tbl = gr.Dataframe(_ledger_view(_init_asof, _init_window), label="Action ledger", interactive=False, wrap=True)
            gr.HTML('<div class="gl-sec">Log a new action</div>')
            with gr.Row():
                f_title = gr.Textbox(label="What was done", placeholder="e.g. Occupancy sensors in corridors", scale=3)
                f_bld = gr.Dropdown(BLDS, value=BLDS[0], label="Building", scale=1)
                f_res = gr.Dropdown(list(RESOURCES), value="electricity", label="Resource", scale=1)
            with gr.Row():
                f_date = gr.Textbox(label="Date (YYYY-MM-DD)", value="2017-10-16", scale=1)
                f_cost = gr.Number(label="Cost (₹)", value=0, scale=1)
                f_claim = gr.Number(label="Claimed saving (%)", value=10, scale=1)
                f_notes = gr.Textbox(label="Notes", scale=2)
            log_btn = gr.Button("Log action", variant="primary")
            log_msg = gr.HTML()

        with gr.Tab("Verify Savings", id="verify"):
            gr.HTML(intro("<b>Did it really save anything?</b> A model trained on the year before the action, using "
                          "<b>untreated comparison buildings</b>, weather and day-of-week, predicts what would have happened "
                          "without it. A <b>placebo test</b> repeats this at dozens of dates when nothing happened, to measure how "
                          "big a fake saving noise alone can produce. A saving is VERIFIED only if the whole 95% interval is above zero."))
            with gr.Row():
                vf_dd = gr.Dropdown(_init_actions, value="A1", label="Action", scale=4)
                vf_btn = gr.Button("Verify", variant="primary", scale=1)
            vf_card = gr.HTML(_vf[0])
            vf_comp = gr.HTML(_vf[3])
            with gr.Row():
                vf_plot1 = gr.Plot(_vf[1])
                vf_plot2 = gr.Plot(_vf[2])

        with gr.Tab("Evidence", id="evidence"):
            gr.HTML(intro("<b>Why you can trust the numbers.</b> We injected problems and interventions with KNOWN truth into "
                          "real meter data. Detector thresholds were tuned on a separate development set; the results below "
                          "are on held-out data. Every non-injected alert counts against us, even though some are real events "
                          "in the original data."))
            ev_cards = gr.HTML(_ev[0])
            ev_plot = gr.Plot(_ev[1])
            gr.HTML('<div class="gl-sec">Scenario actions — claimed vs. true vs. naive vs. GreenLedger</div>')
            gr.Dataframe(_ev[2], interactive=False, wrap=True)
            with gr.Row():
                with gr.Column():
                    gr.HTML('<div class="gl-sec">Detection by problem type (held-out)</div>')
                    gr.Dataframe(_ev[3], interactive=False)
                with gr.Column():
                    gr.HTML('<div class="gl-sec">Baseline accuracy as used (out-of-sample, Jul–Dec 2017)</div>')
                    gr.Dataframe(_ev[5], interactive=False)
                    gr.HTML('<div class="gl-note">Bias (NMBE) near 0% = no systematic over/under-prediction. Hourly '
                            'scatter is what the σ-based alert thresholds are scaled to; real hourly water use is '
                            'spiky, which is why water alerts use night-flow and 6-hour tests instead of single hours.</div>')
            gr.HTML('<div class="gl-sec">Every injected problem and whether it was caught</div>')
            gr.Dataframe(_ev[4], interactive=False)
            gr.HTML('<div class="gl-sec">Data provenance — what is real, synthetic or injected</div>')
            gr.Dataframe(PROVENANCE, interactive=False, wrap=True)
            gr.File(value=LABELLED_ZIP, label="Download the labelled sample dataset (CSV + truth tables)")

        with gr.Tab("Report", id="report"):
            gr.HTML(intro("<b>Exportable report</b> for the selected week: printable HTML (open → Print → Save as PDF), "
                          "Markdown, CSV ledger and JSON. Only verified savings are counted, at their lower bound."))
            rep_btn = gr.Button("Generate report", variant="primary")
            rep_files = gr.File(label="Downloads", file_count="multiple")
            rep_md = gr.Markdown()

        with gr.Tab("E-Waste (extension)", id="ewaste"):
            gr.HTML(intro("<b>Extension:</b> device inventory screened for end-of-life, with a recovery path for each flagged "
                          "device. These CO₂ figures are <b>assumption-based estimates</b> and are never added to measured savings."))
            gr.HTML(_ew[0])
            gr.Plot(_ew[1])
            gr.Dataframe(_ew[2], interactive=False, wrap=True)

    gr.HTML(f'<div class="gl-foot"><span>GreenLedger · counterfactual verification · placebo-in-time CIs</span>'
            f'<span>{EF_SOURCE}</span></div>')

    # ---------- wiring ----------
    def _refresh_all(as_of_v, window, res_v, st_v):
        c, f, s = ui_overview(as_of_v, window)
        t, dd, fig, ex = ui_alerts(as_of_v, res_v, st_v)
        return c, f, s, t, dd, fig, ex, _ledger_view(as_of_v, window)

    _outs = [ov_cards, ov_plot, ov_score, al_tbl, al_dd, al_plot, al_expl, reg_tbl]
    asof_dd.change(_refresh_all, [asof_dd, window_rb, res_f, st_f], _outs)
    window_rb.change(_refresh_all, [asof_dd, window_rb, res_f, st_f], _outs)
    res_f.change(ui_alerts, [asof_dd, res_f, st_f], [al_tbl, al_dd, al_plot, al_expl])
    st_f.change(ui_alerts, [asof_dd, res_f, st_f], [al_tbl, al_dd, al_plot, al_expl])
    al_dd.change(ui_alert_detail, [al_dd], [al_plot, al_expl])
    _dec_outs = [al_msg, al_tbl, al_dd, al_plot, al_expl, dec_tbl]
    conf_btn.click(lambda a, n, x, w, r, s: ui_decide(a, "CONFIRMED", n, x, r, s),
                   [al_dd, note_tb, asof_dd, window_rb, res_f, st_f], _dec_outs)
    dism_btn.click(lambda a, n, x, w, r, s: ui_decide(a, "DISMISSED", n, x, r, s),
                   [al_dd, note_tb, asof_dd, window_rb, res_f, st_f], _dec_outs)
    act_btn.click(ui_alert_to_action, [al_dd], [al_msg, f_title, f_bld, f_res, f_date, tabs]).then(
        _decision_log, None, dec_tbl)
    log_btn.click(ui_log_action, [f_title, f_bld, f_res, f_date, f_cost, f_claim, f_notes, asof_dd, window_rb],
                  [log_msg, reg_tbl, vf_dd])
    vf_btn.click(ui_verify, [vf_dd, asof_dd, window_rb], [vf_card, vf_plot1, vf_plot2, vf_comp])
    vf_dd.change(ui_verify, [vf_dd, asof_dd, window_rb], [vf_card, vf_plot1, vf_plot2, vf_comp])
    rep_btn.click(ui_report, [asof_dd, window_rb], [rep_md, rep_files])

print("UI built.")

## Results summary

In [ ]:
# ============================================================
# 13b · RESULTS SUMMARY (numbers computed above — nothing hard-coded)
# ============================================================
_led = ledger(DATA_END)
_tot = verified_totals(_led)
print("=" * 78)
print("GREENLEDGER — RESULTS")
print("=" * 78)
print(f"Data: REAL hourly electricity + water, {len(BLDS)} buildings, site '{SITE}', {IDX.min():%b %Y}–{IDX.max():%b %Y}")
print(f"\nSAVINGS VERIFIER (held-out validation, {MV_VALID_SUMMARY['cases']} injected effects in real data)")
print(f"  95% interval coverage            : {MV_VALID_SUMMARY['ci_coverage']:.0%}")
print(f"  False savings claims, zero effect : {MV_VALID_SUMMARY['false_claims_on_zero_effect']}   "
      f"(naive before/after: {MV_VALID_SUMMARY['naive_false_claims_on_zero_effect']})")
print(f"\nALERT DETECTOR (held-out labelled set; thresholds tuned on a separate dev set)")
print(f"  Problems caught                   : {DETECT_SUMMARY['detected']} of {DETECT_SUMMARY['injected']}")
print(f"  Other alerts per meter per month  : {DETECT_SUMMARY['other_alerts_per_series_month']}")
print(f"\nSCENARIO ACTIONS")
for a in ACTIONS_TRUTH:
    r = MV_RESULTS[a["action_id"]]
    print(f"  {a['action_id']} {a['title'][:40]:40s} claimed {a['claimed_pct']:4.0f}% | true {true_effect_pct(a):5.1f}% | "
          f"naive {r['naive_pct']:5.1f}% | GreenLedger {r['estimate_pct']:5.1f}% [{r['ci_lo_pct']:.1f}, {r['ci_hi_pct']:.1f}] → {r['verdict']}")
print(f"\nCOUNTED (95% lower bound only): {_tot['electricity']:,.0f} kWh (≥ {_tot['co2_kg'] / 1000:.1f} tCO₂) · "
      f"{_tot['water'] / 1000:,.1f} kL water · {_tot['waste']:,.0f} kg dry waste")

## Launch

In [ ]:
# ============================================================
# 14 · LAUNCH  (Kaggle: turn Internet ON in notebook settings for the public share link)
# ============================================================
if os.environ.get("GL_NO_LAUNCH") != "1":
    try:
        demo.queue().launch(share=True, debug=False)
    except Exception as ex:
        print("Share link unavailable (is Internet enabled?):", ex)
        demo.queue().launch(share=False, debug=False)

## 5-minute demo script

1. **Overview (45 s).** "One view of electricity, water and waste for five real buildings. Each number is compared with what the building normally uses for this weather and day. Colour only appears when a change is statistically unusual, not when a percentage merely looks big. The savings cards count only verified savings, at their lower bound."
2. **Time travel (20 s).** Change *Viewing campus as of* to an early week (July). "Verdicts and alerts only use data available that week. Nothing is claimed before the measurement window ends."
3. **Alerts (60 s).** Open a water *night-flow* alert. "At 3 a.m. a classroom block should use almost no water. This one doesn't: likely a leak. Here's the excess, when it was raised, and what to check." Click **Turn into action**, and the register form fills in.
4. **Verify: the money shot (90 s).**
   - **A1 LED retrofit → VERIFIED.** Point at the placebo histogram: "These are 'savings' our method finds on dates when nothing happened. The real action sits far outside them."
   - **A3 awareness campaign → NOT PROVEN.** "The team claimed 20%. A simple before/after says about 9%. We say not proven, because the whole campus dropped at the same time (term end), and we don't count a rupee of it." *This is the 'without overstating' line, delivered.*
5. **Evidence (45 s).** "We injected problems and interventions with known truth into real meter data. Zero false savings claims on zero-effect actions, versus 6 of 13 for before/after. Thresholds were tuned on a separate development set."
6. **Report (20 s).** Generate the report and open the HTML. "One click: the sustainability office's monthly report, with only defensible numbers."

**Likely judge questions**
- *Why not just use before/after?* Because it confuses season and calendar with impact: see A3, and the 6 of 13 false claims.
- *What if every building gets the retrofit?* Comparison groups disappear, so the verdict falls back to NOT PROVEN with the reason shown. We'd recommend phased roll-outs, which also give a natural control group.
- *Is the data real?* Electricity and water are real BDG2 meters; waste is synthetic and labelled as such. The Provenance table says exactly what is real.
- *Why do some actions say NOT PROVEN when they did work (A4)?* A 2.5% effect is smaller than this campus's normal variation over 6 weeks. We report "not proven", not "no effect". A longer window or sub-metering would settle it.